# **BCI**


In [ ]:
import numpy as np
from scipy.linalg import eigh
from scipy.signal import butter, filtfilt, sosfiltfilt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt

In [ ]:
class CSPProcessor:
    def __init__(self, n_components=6, filter_band=(8, 30)):
        """
        Parameters:
        n_components : Number of CSP components to use (3 from each class)
        filter_band : Frequency band to filter (low, high) in Hz
        """
        self.n_components = n_components
        self.filter_band = filter_band
        self.filters = None
        self.eigenvalues = None

    def bandpass_filter(self, data, fs, order=5):
        """Apply bandpass filter to EEG data"""
        nyq = 0.5 * fs
        low = self.filter_band[0] / nyq
        high = self.filter_band[1] / nyq
        sos = butter(order, [low, high], btype='band', output='sos')

        # Apply filter to each channel
        filtered_data = np.zeros_like(data)
        for ch in range(data.shape[0]):
            filtered_data[ch, :] = sosfiltfilt(sos, data[ch, :])

        return filtered_data

    def compute_covariance(self, trials):
        """Compute normalized covariance matrix for trials"""
        n_channels = trials[0].shape[0]
        cov = np.zeros((n_channels, n_channels))

        for trial in trials:
            # Normalize by trace
            trial_cov = np.dot(trial, trial.T) / np.trace(np.dot(trial, trial.T))
            cov += trial_cov

        return cov / len(trials)

    def fit(self, trials_class1, trials_class2):
        """
        Fit CSP filters

        Parameters:
        trials_class1 : Each array is (n_channels, n_samples) for class 1
        trials_class2 : Each array is (n_channels, n_samples) for class 2
        """
        # Compute covariance matrices for each class
        cov1 = self.compute_covariance(trials_class1)
        cov2 = self.compute_covariance(trials_class2)

        # Solve generalized eigenvalue problem
        eigenvalues, eigenvectors = eigh(cov1, cov1 + cov2)

        # Sort by eigenvalues (descending)
        idx = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]

        # Select top and bottom components
        n_comp_per_class = self.n_components // 2
        selected_filters = np.hstack([
            eigenvectors[:, :n_comp_per_class],  # Top components (class 1)
            eigenvectors[:, -n_comp_per_class:]  # Bottom components (class 2)
        ])

        self.filters = selected_filters.T  # Shape: (n_components, n_channels)
        self.eigenvalues = eigenvalues

        return self

    def transform(self, trial):
        """
        Apply CSP transformation to a single trial

        Parameters:
        -----------
        trial : array (n_channels, n_samples)

        Returns:
        --------
        features : array (n_components,)
            Log variance of CSP-filtered signals
        """
        if self.filters is None:
            raise ValueError("CSP filters not fitted. Call fit() first.")

        # Apply spatial filters
        filtered = np.dot(self.filters, trial)  # (n_components, n_samples)

        # Compute log variance for each component
        variances = np.var(filtered, axis=1)
        features = np.log(variances + 1e-8)  # Add small constant for stability

        return features

    def fit_transform(self, trials_class1, trials_class2):
        """Fit and transform training data"""
        self.fit(trials_class1, trials_class2)

        # Transform all trials
        features_class1 = np.array([self.transform(t) for t in trials_class1])
        features_class2 = np.array([self.transform(t) for t in trials_class2])

        return features_class1, features_class2

In [ ]:
def preprocess_eeg_data(all_data_dict, valid_channels=None):
    """
    Preprocess and organize EEG data for CSP

    Parameters:
    all_data_dict : Dictionary containing session and run data
    valid_channels : List of valid channel names to keep

    Returns:
    --------
    processed_data : Organized trials by session and task
    """
    if valid_channels is None:
        valid_channels = ['F3','F4','Fc5','Fc3','Fc1','Fcz','Fc2','Fc4','Fc6',
                         'T7','C5','C3','C1','Cz','C2','C4','C6','T8','Cp5',
                         'Cp3','Cp1','Cpz','Cp2','Cp4','Cp6','P3','P4']

    processed_data = {}

    for session, tasks in all_data_dict.items():
        processed_data[session] = {}

        for task, runs in tasks.items():
            trials_class1 = []  # Right (LR) or Up (UD)
            trials_class2 = []  # Left (LR) or Down (UD)

            for runData in runs:
                # Get channel indices for valid channels
                labels = runData.label if hasattr(runData, 'label') else []
                if isinstance(labels, np.ndarray):
                    labels = labels.tolist()

                valid_idx = [i for i, label in enumerate(labels)
                           if label in valid_channels]

                fs = runData.fs

                # Process each trial
                for i, trial in enumerate(runData.trials):
                    if runData.outcome[i] == 1:  # Only successful trials
                        # Select valid channels
                        trial_data = trial[valid_idx, :]

                        # Separate by target class
                        if runData.target[i] == 1:
                            trials_class1.append(trial_data)
                        elif runData.target[i] == 2:
                            trials_class2.append(trial_data)

            processed_data[session][task] = {
                'class1': trials_class1,
                'class2': trials_class2,
                'fs': fs
            }

    return processed_data

In [ ]:
def apply_csp_pipeline(processed_data, filter_band=(8, 30), n_components=6):
    """
    Apply CSP to preprocessed data

    Parameters:
    processed_data : Output from preprocess_eeg_data
    filter_band : Frequency band for filtering (low, high) Hz
    n_components : Number of CSP components

    Returns:
    csp_features : CSP features for each session and task
    csp_models : Fitted CSP models
    """
    csp_features = {}
    csp_models = {}

    for session, tasks in processed_data.items():
        csp_features[session] = {}
        csp_models[session] = {}

        for task, data in tasks.items():
            print(f"\nProcessing {session} - {task}")
            print(f"Class 1 trials: {len(data['class1'])}")
            print(f"Class 2 trials: {len(data['class2'])}")

            # Initialize CSP
            csp = CSPProcessor(n_components=n_components, filter_band=filter_band)

            # Apply bandpass filter to all trials
            fs = data['fs']
            trials_c1_filtered = [csp.bandpass_filter(t, fs) for t in data['class1']]
            trials_c2_filtered = [csp.bandpass_filter(t, fs) for t in data['class2']]

            # Fit and transform
            features_c1, features_c2 = csp.fit_transform(trials_c1_filtered,
                                                         trials_c2_filtered)

            # Store results
            csp_features[session][task] = {
                'features': np.vstack([features_c1, features_c2]),
                'labels': np.hstack([np.ones(len(features_c1)),
                                    np.zeros(len(features_c2))])
            }
            csp_models[session][task] = csp

            print(f"Features shape: {csp_features[session][task]['features'].shape}")

    return csp_features, csp_models

In [ ]:
def create_train_test_split(processed_data, test_size=0.2, random_state=42):
    """
    Create train/test splits for each session and task

    Parameters:
    processed_data : Output from preprocess_eeg_data
    test_size : Proportion of data for testing
    random_state : Random seed for reproducibility

    Returns:
    --------
    train_data, test_data
    """
    np.random.seed(random_state)
    train_data = {}
    test_data = {}

    for session, tasks in processed_data.items():
        train_data[session] = {}
        test_data[session] = {}

        for task, data in tasks.items():
            n_class1 = len(data['class1'])
            n_class2 = len(data['class2'])

            # Create indices for splitting
            idx_c1 = np.random.permutation(n_class1)
            idx_c2 = np.random.permutation(n_class2)

            split_c1 = int(n_class1 * (1 - test_size))
            split_c2 = int(n_class2 * (1 - test_size))

            # Split class 1
            train_c1 = [data['class1'][i] for i in idx_c1[:split_c1]]
            test_c1 = [data['class1'][i] for i in idx_c1[split_c1:]]

            # Split class 2
            train_c2 = [data['class2'][i] for i in idx_c2[:split_c2]]
            test_c2 = [data['class2'][i] for i in idx_c2[split_c2:]]

            train_data[session][task] = {
                'class1': train_c1,
                'class2': train_c2,
                'fs': data['fs']
            }

            test_data[session][task] = {
                'class1': test_c1,
                'class2': test_c2,
                'fs': data['fs']
            }

            print(f"{session}-{task}: Train C1={len(train_c1)}, C2={len(train_c2)} | "
                  f"Test C1={len(test_c1)}, C2={len(test_c2)}")

    return train_data, test_data

In [ ]:
def apply_csp_with_split(train_data, test_data, filter_band=(8, 30), n_components=6):
    """
    Apply CSP to train data and transform test data

    Returns:
    train_features, test_features : CSP features for train and test sets
    csp_models : Fitted CSP models
    """
    train_features = {}
    test_features = {}
    csp_models = {}

    for session in train_data.keys():
        train_features[session] = {}
        test_features[session] = {}
        csp_models[session] = {}

        for task in train_data[session].keys():
            print(f"\nProcessing {session} - {task}")

            # Initialize CSP
            csp = CSPProcessor(n_components=n_components, filter_band=filter_band)

            # Get training data
            train_d = train_data[session][task]
            test_d = test_data[session][task]
            fs = train_d['fs']

            # Apply bandpass filter to training trials
            train_c1_filt = [csp.bandpass_filter(t, fs) for t in train_d['class1']]
            train_c2_filt = [csp.bandpass_filter(t, fs) for t in train_d['class2']]

            # Fit CSP on training data
            feat_c1_train, feat_c2_train = csp.fit_transform(train_c1_filt, train_c2_filt)

            # Apply bandpass filter to test trials
            test_c1_filt = [csp.bandpass_filter(t, fs) for t in test_d['class1']]
            test_c2_filt = [csp.bandpass_filter(t, fs) for t in test_d['class2']]

            # Transform test data using fitted CSP
            feat_c1_test = np.array([csp.transform(t) for t in test_c1_filt])
            feat_c2_test = np.array([csp.transform(t) for t in test_c2_filt])

            # Store results
            train_features[session][task] = {
                'features': np.vstack([feat_c1_train, feat_c2_train]),
                'labels': np.hstack([np.ones(len(feat_c1_train)),
                                    np.zeros(len(feat_c2_train))])
            }

            test_features[session][task] = {
                'features': np.vstack([feat_c1_test, feat_c2_test]),
                'labels': np.hstack([np.ones(len(feat_c1_test)),
                                    np.zeros(len(feat_c2_test))])
            }

            csp_models[session][task] = csp

            print(f"Train features: {train_features[session][task]['features'].shape}")
            print(f"Test features: {test_features[session][task]['features'].shape}")

    return train_features, test_features, csp_models

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [ ]:
class CSPDataset(Dataset):
    """PyTorch Dataset for CSP features"""
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [ ]:
class CNNModel_CSP(nn.Module):
    """
    Input: (batch_size, n_components, sequence_length)
    Output: (batch_size, 2) for binary classification
    """
    def __init__(self, n_components=6, sequence_length=128, n_classes=2):
        super().__init__()

        self.network = nn.Sequential(
            # ----- CNN Layer 1 -----
            nn.Conv1d(in_channels=n_components, out_channels=64,
                     kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.MaxPool1d(kernel_size=2),   # L: 128 -> 64

            # ----- CNN Layer 2 -----
            nn.Conv1d(in_channels=64, out_channels=128,
                     kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.MaxPool1d(kernel_size=2),   # L: 64 -> 32

            # ----- CNN Layer 3 -----
            nn.Conv1d(in_channels=128, out_channels=256,
                     kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.MaxPool1d(kernel_size=2),   # L: 32 -> 16

            nn.Flatten(),
            # Calculate flattened size: 256 channels * 16 time points
            nn.Linear(256 * 16, 120),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(120, n_classes)
        )

    def forward(self, xb):
        return self.network(xb)

In [ ]:
def expand_csp_features(features, target_length=128):
    """
    Expand CSP features to sequence for CNN

    Parameters:
    features : array (n_samples, n_components)
    target_length : Desired sequence length

    Returns:
    expanded : array (n_samples, n_components, target_length)
    """
    n_samples, n_components = features.shape
    expanded = np.zeros((n_samples, n_components, target_length))

    # Replicate features across time dimension
    for i in range(n_samples):
        for j in range(n_components):
            expanded[i, j, :] = features[i, j]

    return expanded

In [ ]:
def train_cnn_model(train_features, test_features, session, task,
                   n_epochs=50, batch_size=32, lr=0.001, device='cuda'):
    """
    Train CNN model on CSP features
    """
    # Prepare data
    X_train = expand_csp_features(train_features[session][task]['features'])
    y_train = train_features[session][task]['labels'].astype(int)

    X_test = expand_csp_features(test_features[session][task]['features'])
    y_test = test_features[session][task]['labels'].astype(int)

    # Create datasets
    train_dataset = CSPDataset(X_train, y_train)
    test_dataset = CSPDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Initialize model
    n_components = X_train.shape[1]
    model = CNNModel_CSP(n_components=n_components, n_classes=2)
    model = model.to(device)

    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Training loop
    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

    for epoch in range(n_epochs):
        # Training
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        # Testing
        model.eval()
        test_loss = 0
        test_correct = 0
        test_total = 0

        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                test_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                test_total += labels.size(0)
                test_correct += (predicted == labels).sum().item()

        # Record history
        history['train_loss'].append(train_loss / len(train_loader))
        history['train_acc'].append(100 * train_correct / train_total)
        history['test_loss'].append(test_loss / len(test_loader))
        history['test_acc'].append(100 * test_correct / test_total)

        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{n_epochs}] "
                  f"Train Loss: {history['train_loss'][-1]:.4f}, "
                  f"Train Acc: {history['train_acc'][-1]:.2f}%, "
                  f"Test Loss: {history['test_loss'][-1]:.4f}, "
                  f"Test Acc: {history['test_acc'][-1]:.2f}%")

    return model, history


# ==================== VISUALIZATION FUNCTIONS ====================

import seaborn as sns
from mpl_toolkits.axes_grid1 import make_axes_locatable
from sklearn.metrics import confusion_matrix

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10


def plot_raw_eeg_sample(runData, trial_idx=0, duration=2.0):
    """Visualize raw EEG from a single trial"""
    trial = runData.trials[trial_idx]
    fs = runData.fs

    # Use actual trial length or requested duration, whichever is smaller
    actual_duration = trial.shape[1] / fs
    duration = min(duration, actual_duration)
    n_samples = int(duration * fs)
    time = np.arange(n_samples) / fs

    # Select subset of channels for clarity
    channels_to_plot = [0, 5, 10, 15, 20, 25]
    channel_labels = [runData.label[i] for i in channels_to_plot]

    fig, axes = plt.subplots(len(channels_to_plot), 1, figsize=(14, 10), sharex=True)
    fig.suptitle(f'Raw EEG - Trial {trial_idx} (Target: {runData.target[trial_idx]}, Duration: {duration:.2f}s)',
                 fontsize=16, fontweight='bold')

    for i, (ch_idx, ax) in enumerate(zip(channels_to_plot, axes)):
        ax.plot(time, trial[ch_idx, :n_samples], linewidth=0.8, color='steelblue')
        ax.set_ylabel(f'{channel_labels[i]}\n(µV)', fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.axvspan(0, min(0.5, duration), alpha=0.1, color='green', label='Baseline' if i == 0 else '')
        if duration > 0.5:
            ax.axvspan(0.5, duration, alpha=0.1, color='orange', label='Motor Imagery' if i == 0 else '')

    axes[-1].set_xlabel('Time (seconds)', fontsize=12)
    axes[0].legend(loc='upper right')
    plt.tight_layout()
    plt.show()
    return fig


def plot_frequency_spectrum(runData, trial_indices=[0, 1], show_bands=True):
    """Show frequency content of EEG signals"""
    from scipy import signal as sig
    fs = runData.fs

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle('Frequency Spectrum Analysis', fontsize=16, fontweight='bold')

    for idx, trial_idx in enumerate(trial_indices):
        trial = runData.trials[trial_idx]
        target = runData.target[trial_idx]

        freqs, psd = sig.welch(trial[13, :], fs, nperseg=512)

        axes[idx].semilogy(freqs, psd, linewidth=2, color='navy', alpha=0.7)
        axes[idx].set_xlabel('Frequency (Hz)', fontsize=11)
        axes[idx].set_ylabel('Power Spectral Density (µV²/Hz)', fontsize=11)
        axes[idx].set_title(f'Trial {trial_idx} - Target: {"Right/Up" if target == 1 else "Left/Down"}',
                           fontsize=12)
        axes[idx].set_xlim([0, 50])
        axes[idx].grid(True, alpha=0.3)

        if show_bands:
            axes[idx].axvspan(8, 13, alpha=0.2, color='green', label='Mu (8-13 Hz)')
            axes[idx].axvspan(13, 30, alpha=0.2, color='orange', label='Beta (13-30 Hz)')
            axes[idx].legend(loc='upper right')

    plt.tight_layout()
    plt.show()
    return fig


def plot_filtering_effect(trial, fs, filter_band=(8, 30), channel_idx=None):
    """Show effect of bandpass filtering"""
    # If no channel specified, use the middle channel
    if channel_idx is None:
        channel_idx = trial.shape[0] // 2

    # Make sure channel_idx is valid
    if channel_idx >= trial.shape[0]:
        channel_idx = trial.shape[0] // 2
        print(f"Warning: Channel index adjusted to {channel_idx} (middle channel)")

    nyq = 0.5 * fs
    low = filter_band[0] / nyq
    high = filter_band[1] / nyq
    sos = butter(5, [low, high], btype='band', output='sos')
    filtered = sosfiltfilt(sos, trial[channel_idx, :])

    time = np.arange(len(trial[channel_idx, :])) / fs

    # Use actual trial length or 2 seconds, whichever is smaller
    actual_duration = len(trial[channel_idx, :]) / fs
    duration = min(2.0, actual_duration)
    n_samples = int(duration * fs)

    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    fig.suptitle(f'Bandpass Filtering Effect ({filter_band[0]}-{filter_band[1]} Hz) - Channel {channel_idx}',
                 fontsize=16, fontweight='bold')

    axes[0].plot(time[:n_samples], trial[channel_idx, :n_samples],
                linewidth=0.8, color='steelblue')
    axes[0].set_ylabel('Raw EEG (µV)', fontsize=11)
    axes[0].set_title('Original Signal', fontsize=12)
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(time[:n_samples], filtered[:n_samples],
                linewidth=0.8, color='darkgreen')
    axes[1].set_ylabel('Filtered EEG (µV)', fontsize=11)
    axes[1].set_title('After Bandpass Filter (Mu + Beta bands)', fontsize=12)
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(time[:n_samples], trial[channel_idx, :n_samples],
                linewidth=0.8, color='steelblue', alpha=0.5, label='Original')
    axes[2].plot(time[:n_samples], filtered[:n_samples],
                linewidth=0.8, color='darkgreen', label='Filtered')
    axes[2].set_xlabel('Time (seconds)', fontsize=11)
    axes[2].set_ylabel('Amplitude (µV)', fontsize=11)
    axes[2].set_title('Comparison', fontsize=12)
    axes[2].legend(loc='upper right')
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return fig


def plot_csp_patterns(csp_model, channel_labels=None):
    """Visualize CSP spatial filters"""
    filters = csp_model.filters
    n_channels = filters.shape[1]

    # If no labels provided or mismatch, create generic labels
    if channel_labels is None or len(channel_labels) != n_channels:
        channel_labels = [f'Ch{i+1}' for i in range(n_channels)]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(f'CSP Spatial Patterns (Top 3 for each class) - {n_channels} channels',
                 fontsize=16, fontweight='bold')

    axes = axes.flatten()

    for i in range(6):
        pattern = filters[i, :]
        colors = ['red' if p < 0 else 'blue' for p in pattern]
        axes[i].bar(range(len(pattern)), pattern, color=colors, alpha=0.7)
        axes[i].set_xticks(range(len(pattern)))
        axes[i].set_xticklabels(channel_labels, rotation=90, fontsize=8)
        axes[i].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
        axes[i].set_ylabel('Weight', fontsize=10)
        axes[i].grid(True, alpha=0.3, axis='y')

        if i < 3:
            axes[i].set_title(f'CSP Component {i+1} (Class 1)', fontsize=11, color='darkred')
        else:
            axes[i].set_title(f'CSP Component {i+1} (Class 2)', fontsize=11, color='darkblue')

    plt.tight_layout()
    plt.show()
    return fig


def plot_eigenvalue_spectrum(csp_model):
    """Plot CSP eigenvalue spectrum"""
    eigenvalues = csp_model.eigenvalues

    fig, ax = plt.subplots(1, 1, figsize=(12, 6))

    ax.plot(range(len(eigenvalues)), eigenvalues, 'o-',
           linewidth=2, markersize=8, color='navy')
    ax.axhline(y=0.5, color='red', linestyle='--', linewidth=2,
              label='Equal variance (0.5)')

    n_comp = 3
    ax.scatter(range(n_comp), eigenvalues[:n_comp],
              s=200, c='red', marker='o', edgecolors='black',
              linewidth=2, label='Selected (Class 1)', zorder=5)
    ax.scatter(range(len(eigenvalues)-n_comp, len(eigenvalues)),
              eigenvalues[-n_comp:],
              s=200, c='blue', marker='o', edgecolors='black',
              linewidth=2, label='Selected (Class 2)', zorder=5)

    ax.set_xlabel('Component Index', fontsize=12)
    ax.set_ylabel('Eigenvalue (Variance Ratio)', fontsize=12)
    ax.set_title('CSP Eigenvalue Spectrum\n(Higher = More discriminative for Class 1)',
                fontsize=14, fontweight='bold')
    ax.legend(loc='right', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([-0.1, 1.1])

    plt.tight_layout()
    plt.show()
    return fig


def plot_csp_features_distribution(train_features, session='Session1', task='LR'):
    """Visualize distribution of CSP features for both classes"""
    features = train_features[session][task]['features']
    labels = train_features[session][task]['labels']

    class1_features = features[labels == 1]
    class2_features = features[labels == 0]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(f'CSP Feature Distributions - {session} {task}',
                 fontsize=16, fontweight='bold')

    axes = axes.flatten()

    for i in range(6):
        axes[i].hist(class1_features[:, i], bins=20, alpha=0.6,
                    color='red', label='Class 1 (Right/Up)', density=True)
        axes[i].hist(class2_features[:, i], bins=20, alpha=0.6,
                    color='blue', label='Class 2 (Left/Down)', density=True)
        axes[i].set_xlabel(f'CSP Feature {i+1} (log variance)', fontsize=10)
        axes[i].set_ylabel('Density', fontsize=10)
        axes[i].set_title(f'Component {i+1}', fontsize=11)
        axes[i].legend(loc='upper right', fontsize=9)
        axes[i].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return fig


def plot_csp_features_2d(train_features, session='Session1', task='LR'):
    """2D scatter plot of most discriminative CSP features"""
    features = train_features[session][task]['features']
    labels = train_features[session][task]['labels']

    class1_features = features[labels == 1]
    class2_features = features[labels == 0]

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle(f'CSP Feature Space - {session} {task}',
                 fontsize=16, fontweight='bold')

    axes[0].scatter(class1_features[:, 0], class1_features[:, 1],
                   c='red', alpha=0.6, s=50, label='Class 1 (Right/Up)', edgecolors='black')
    axes[0].scatter(class2_features[:, 0], class2_features[:, 1],
                   c='blue', alpha=0.6, s=50, label='Class 2 (Left/Down)', edgecolors='black')
    axes[0].set_xlabel('CSP Component 1', fontsize=12)
    axes[0].set_ylabel('CSP Component 2', fontsize=12)
    axes[0].set_title('Components 1 vs 2', fontsize=12)
    axes[0].legend(loc='best')
    axes[0].grid(True, alpha=0.3)

    axes[1].scatter(class1_features[:, -2], class1_features[:, -1],
                   c='red', alpha=0.6, s=50, label='Class 1 (Right/Up)', edgecolors='black')
    axes[1].scatter(class2_features[:, -2], class2_features[:, -1],
                   c='blue', alpha=0.6, s=50, label='Class 2 (Left/Down)', edgecolors='black')
    axes[1].set_xlabel('CSP Component 5', fontsize=12)
    axes[1].set_ylabel('CSP Component 6', fontsize=12)
    axes[1].set_title('Components 5 vs 6', fontsize=12)
    axes[1].legend(loc='best')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return fig


def plot_feature_expansion():
    """Illustrate feature expansion process"""
    csp_features = np.array([2.5, 1.8, 2.1, -0.3, -0.5, -0.2])
    expanded = np.tile(csp_features.reshape(-1, 1), (1, 128))

    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    fig.suptitle('Feature Expansion for CNN Input', fontsize=16, fontweight='bold')

    axes[0].bar(range(6), csp_features, color=['red', 'red', 'red', 'blue', 'blue', 'blue'],
               alpha=0.7, edgecolor='black')
    axes[0].set_xlabel('CSP Component', fontsize=12)
    axes[0].set_ylabel('Log Variance', fontsize=12)
    axes[0].set_title('Original CSP Features (6 values)', fontsize=12)
    axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
    axes[0].grid(True, alpha=0.3, axis='y')
    axes[0].set_xticks(range(6))
    axes[0].set_xticklabels([f'CSP-{i+1}' for i in range(6)])

    im = axes[1].imshow(expanded, aspect='auto', cmap='RdBu_r',
                       interpolation='nearest', vmin=-3, vmax=3)
    axes[1].set_xlabel('Time Steps (Replicated)', fontsize=12)
    axes[1].set_ylabel('CSP Component', fontsize=12)
    axes[1].set_title('Expanded Features (6 × 128) - Ready for CNN', fontsize=12)
    axes[1].set_yticks(range(6))
    axes[1].set_yticklabels([f'CSP-{i+1}' for i in range(6)])

    divider = make_axes_locatable(axes[1])
    cax = divider.append_axes("right", size="2%", pad=0.1)
    plt.colorbar(im, cax=cax, label='Log Variance')

    plt.tight_layout()
    plt.show()
    return fig


def plot_training_history(history, session='Session1', task='LR'):
    """Plot training and validation metrics over epochs"""
    epochs = range(1, len(history['train_loss']) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle(f'Training History - {session} {task}', fontsize=16, fontweight='bold')

    axes[0].plot(epochs, history['train_loss'], 'o-',
                linewidth=2, markersize=4, color='blue', label='Train Loss')
    axes[0].plot(epochs, history['test_loss'], 's-',
                linewidth=2, markersize=4, color='red', label='Test Loss')
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title('Loss vs Epoch', fontsize=13)
    axes[0].legend(loc='upper right', fontsize=11)
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history['train_acc'], 'o-',
                linewidth=2, markersize=4, color='blue', label='Train Accuracy')
    axes[1].plot(epochs, history['test_acc'], 's-',
                linewidth=2, markersize=4, color='red', label='Test Accuracy')
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Accuracy (%)', fontsize=12)
    axes[1].set_title('Accuracy vs Epoch', fontsize=13)
    axes[1].legend(loc='lower right', fontsize=11)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim([0, 105])

    final_train_acc = history['train_acc'][-1]
    final_test_acc = history['test_acc'][-1]
    axes[1].text(0.02, 0.98,
                f'Final Train: {final_train_acc:.2f}%\nFinal Test: {final_test_acc:.2f}%',
                transform=axes[1].transAxes, fontsize=11, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.tight_layout()
    plt.show()
    return fig


def plot_confusion_matrix_bci(model, test_features, session='Session1', task='LR', device='cuda'):
    """Plot confusion matrix for test set predictions"""
    X_test = expand_csp_features(test_features[session][task]['features'])
    y_test = test_features[session][task]['labels'].astype(int)

    model.eval()
    X_test_tensor = torch.FloatTensor(X_test).to(device)

    with torch.no_grad():
        outputs = model(X_test_tensor)
        _, predictions = torch.max(outputs, 1)

    predictions = predictions.cpu().numpy()
    cm = confusion_matrix(y_test, predictions)

    fig, ax = plt.subplots(1, 1, figsize=(8, 6))

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Left/Down', 'Right/Up'],
                yticklabels=['Left/Down', 'Right/Up'],
                cbar_kws={'label': 'Count'}, ax=ax)

    ax.set_xlabel('Predicted Label', fontsize=12)
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_title(f'Confusion Matrix - {session} {task}\nTest Accuracy: {100 * cm.diagonal().sum() / cm.sum():.2f}%',
                fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.show()
    return fig, cm


def plot_pipeline_summary(all_results):
    """Summary of results across all sessions and tasks"""
    sessions = list(all_results.keys())
    tasks = ['LR', 'UD']

    data = []
    for session in sessions:
        for task in tasks:
            if task in all_results[session]:
                data.append({
                    'session': session,
                    'task': task,
                    'accuracy': all_results[session][task]['test_acc'][-1]
                })

    fig, ax = plt.subplots(1, 1, figsize=(10, 6))

    x = np.arange(len(data))
    colors = ['steelblue' if d['task'] == 'LR' else 'coral' for d in data]
    bars = ax.bar(x, [d['accuracy'] for d in data], color=colors, alpha=0.8, edgecolor='black')

    ax.set_xlabel('Session - Task', fontsize=12)
    ax.set_ylabel('Test Accuracy (%)', fontsize=12)
    ax.set_title('BCI Classification Performance Summary', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([f"{d['session']}\n{d['task']}" for d in data])
    ax.set_ylim([0, 105])
    ax.axhline(y=50, color='red', linestyle='--', linewidth=2, label='Chance Level (50%)')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3, axis='y')

    for bar, d in zip(bars, data):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 1,
               f'{d["accuracy"]:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.show()
    return fig

In [ ]:

# Example usage:
if __name__ == "__main__":
    # Step 1: Preprocess and split data
    print("\n" + "="*70)
    print("STEP 1: PREPROCESSING AND SPLITTING DATA")
    print("="*70)
    processed_data = preprocess_eeg_data(all_data_dict)
    train_data, test_data = create_train_test_split(processed_data, test_size=0.2)

    # Step 2: Apply CSP with train/test split
    print("\n" + "="*70)
    print("STEP 2: APPLYING CSP TO TRAIN AND TEST DATA")
    print("="*70)
    train_features, test_features, csp_models = apply_csp_with_split(
        train_data, test_data,
        filter_band=(8, 30),
        n_components=6
    )

    # ===== VISUALIZATION SECTION =====
    print("\n" + "="*70)
    print("VISUALIZATION SECTION: GENERATING ALL PLOTS")
    print("="*70)

    # Define valid channels
    valid_channels = ['F3','F4','Fc5','Fc3','Fc1','Fcz','Fc2','Fc4','Fc6',
                      'T7','C5','C3','C1','Cz','C2','C4','C6','T8','Cp5',
                      'Cp3','Cp1','Cpz','Cp2','Cp4','Cp6','P3','P4']

    # Generate visualizations for both sessions and both tasks
    for session in ['Session1', 'Session2']:
        for task in ['LR', 'UD']:
            print(f"\n{'='*70}")
            print(f"VISUALIZATIONS FOR {session} - {task}")
            print(f"{'='*70}")

            # Check if this session-task combination exists
            if session not in all_data_dict or task not in all_data_dict[session]:
                print(f"Warning: {session}-{task} not found in data. Skipping...")
                continue

            if len(all_data_dict[session][task]) == 0:
                print(f"Warning: {session}-{task} has no data. Skipping...")
                continue

            # Visualization 1: Raw EEG sample
            print(f"\n=== 1. Raw EEG Sample - {session} {task} ===")
            runData = all_data_dict[session][task][0]
            plot_raw_eeg_sample(runData, trial_idx=5, duration=2.0)

            # Visualization 2: Frequency spectrum
            print(f"\n=== 2. Frequency Spectrum - {session} {task} ===")
            plot_frequency_spectrum(runData, trial_indices=[0, 10])

            # Visualization 3: Filtering effect
            print(f"\n=== 3. Bandpass Filtering Effect - {session} {task} ===")
            if session in processed_data and task in processed_data[session]:
                trial = processed_data[session][task]['class1'][0]
                plot_filtering_effect(trial, fs=300, filter_band=(8, 30))

            # Visualization 4: CSP patterns
            print(f"\n=== 4. CSP Spatial Patterns - {session} {task} ===")
            if session in csp_models and task in csp_models[session]:
                first_run = all_data_dict[session][task][0]
                actual_labels = [label for label in first_run.label if label in valid_channels]
                plot_csp_patterns(csp_models[session][task], actual_labels)

            # Visualization 5: Eigenvalue spectrum
            print(f"\n=== 5. CSP Eigenvalue Spectrum - {session} {task} ===")
            if session in csp_models and task in csp_models[session]:
                plot_eigenvalue_spectrum(csp_models[session][task])

            # Visualization 6: Feature distributions
            print(f"\n=== 6. CSP Feature Distributions - {session} {task} ===")
            if session in train_features and task in train_features[session]:
                plot_csp_features_distribution(train_features, session, task)

            # Visualization 7: 2D feature space
            print(f"\n=== 7. 2D Feature Space - {session} {task} ===")
            if session in train_features and task in train_features[session]:
                plot_csp_features_2d(train_features, session, task)

    # Visualization 8: Feature expansion (only once, it's generic)
    print(f"\n{'='*70}")
    print("=== 8. Feature Expansion Illustration (Generic) ===")
    print(f"{'='*70}")
    plot_feature_expansion()

    # ===== TRAINING SECTION =====
    print("\n" + "="*70)
    print("STEP 3: TRAINING CNN MODELS FOR ALL SESSION-TASK COMBINATIONS")
    print("="*70)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    all_results = {}
    all_models = {}

    for session in train_features.keys():
        all_results[session] = {}
        all_models[session] = {}

        for task in train_features[session].keys():
            print(f"\n{'='*70}")
            print(f"TRAINING: {session} - {task}")
            print(f"{'='*70}")

            # Train the model
            model, history = train_cnn_model(
                train_features, test_features,
                session, task,
                n_epochs=50,
                batch_size=32,
                lr=0.001,
                device=device
            )

            all_results[session][task] = history
            all_models[session][task] = model

            print(f"\n✓ Final Test Accuracy: {history['test_acc'][-1]:.2f}%")

            # Visualization 9: Training history
            print(f"\n=== 9. Training History - {session} {task} ===")
            plot_training_history(history, session, task)

            # Visualization 10: Confusion matrix
            print(f"\n=== 10. Confusion Matrix - {session} {task} ===")
            plot_confusion_matrix_bci(model, test_features, session, task, device)

    # Visualization 11: Overall summary
    print(f"\n{'='*70}")
    print("=== 11. Overall Performance Summary - All Sessions & Tasks ===")
    print(f"{'='*70}")
    plot_pipeline_summary(all_results)

    # ===== FINAL SUMMARY =====
    print("\n" + "="*70)
    print("FINAL RESULTS SUMMARY")
    print("="*70)
    for session in all_results.keys():
        print(f"\n{session}:")
        for task in all_results[session].keys():
            final_acc = all_results[session][task]['test_acc'][-1]
            print(f"  {task}: {final_acc:.2f}% test accuracy")

    print("\n" + "="*70)
    print("PIPELINE COMPLETED SUCCESSFULLY!")
    print("="*70)


STEP 1: PREPROCESSING AND SPLITTING DATA


NameError: name 'all_data_dict' is not defined

In [ ]:
import numpy as np
from scipy.linalg import eigh
from scipy.signal import butter, filtfilt, sosfiltfilt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.io import loadmat

class CSPProcessor:
    """Common Spatial Pattern processor for BCI EEG data"""

    def __init__(self, n_components=6, filter_band=(8, 30)):
        """
        Parameters:
        -----------
        n_components : int
            Number of CSP components to use (3 from each class)
        filter_band : tuple
            Frequency band to filter (low, high) in Hz
        """
        self.n_components = n_components
        self.filter_band = filter_band
        self.filters = None
        self.eigenvalues = None

    def bandpass_filter(self, data, fs, order=5):
        """Apply bandpass filter to EEG data"""
        nyq = 0.5 * fs
        low = self.filter_band[0] / nyq
        high = self.filter_band[1] / nyq
        sos = butter(order, [low, high], btype='band', output='sos')

        # Apply filter to each channel
        filtered_data = np.zeros_like(data)
        for ch in range(data.shape[0]):
            filtered_data[ch, :] = sosfiltfilt(sos, data[ch, :])

        return filtered_data

    def compute_covariance(self, trials):
        """Compute normalized covariance matrix for trials"""
        n_channels = trials[0].shape[0]
        cov = np.zeros((n_channels, n_channels))

        for trial in trials:
            # Normalize by trace
            trial_cov = np.dot(trial, trial.T) / np.trace(np.dot(trial, trial.T))
            cov += trial_cov

        return cov / len(trials)

    def fit(self, trials_class1, trials_class2):
        """
        Fit CSP filters

        Parameters:
        -----------
        trials_class1 : list of arrays
            Each array is (n_channels, n_samples) for class 1
        trials_class2 : list of arrays
            Each array is (n_channels, n_samples) for class 2
        """
        # Compute covariance matrices for each class
        cov1 = self.compute_covariance(trials_class1)
        cov2 = self.compute_covariance(trials_class2)

        # Solve generalized eigenvalue problem
        eigenvalues, eigenvectors = eigh(cov1, cov1 + cov2)

        # Sort by eigenvalues (descending)
        idx = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]

        # Select top and bottom components
        n_comp_per_class = self.n_components // 2
        selected_filters = np.hstack([
            eigenvectors[:, :n_comp_per_class],  # Top components (class 1)
            eigenvectors[:, -n_comp_per_class:]  # Bottom components (class 2)
        ])

        self.filters = selected_filters.T  # Shape: (n_components, n_channels)
        self.eigenvalues = eigenvalues

        return self

    def transform(self, trial):
        """
        Apply CSP transformation to a single trial

        Parameters:
        -----------
        trial : array (n_channels, n_samples)

        Returns:
        --------
        features : array (n_components,)
            Log variance of CSP-filtered signals
        """
        if self.filters is None:
            raise ValueError("CSP filters not fitted. Call fit() first.")

        # Apply spatial filters
        filtered = np.dot(self.filters, trial)  # (n_components, n_samples)

        # Compute log variance for each component
        variances = np.var(filtered, axis=1)
        features = np.log(variances + 1e-8)  # Add small constant for stability

        return features

    def fit_transform(self, trials_class1, trials_class2):
        """Fit and transform training data"""
        self.fit(trials_class1, trials_class2)

        # Transform all trials
        features_class1 = np.array([self.transform(t) for t in trials_class1])
        features_class2 = np.array([self.transform(t) for t in trials_class2])

        return features_class1, features_class2


# ==================== DATA LOADING FUNCTION ====================

def load_bci_data(data_folder_path):
    """
    Load BCI data from .mat files

    Parameters:
    -----------
    data_folder_path : str or Path
        Path to the folder containing Session1 and Session2 folders

    Returns:
    --------
    all_data_dict : dict
        Dictionary with structure: {session: {task: [runData1, runData2, ...]}}
    """
    DATA_FOLDER = Path(data_folder_path)
    session_list = ["Session1", "Session2"]
    tag_list = ["LR", "UD"]

    all_data_dict = {
        session: {tag: [] for tag in tag_list}
        for session in session_list
    }

    print("Loading BCI data files...")
    print("="*70)

    for session in session_list:
        session_path = DATA_FOLDER / session

        if not session_path.exists():
            print(f"Warning: {session} folder not found at {session_path}")
            continue

        for mat_file in session_path.glob("*.mat"):
            tag = mat_file.stem.split("_")[1][:2]  # 'LR' or 'UD'

            if tag not in tag_list:
                print(f"Warning: Unknown tag {tag} in file {mat_file.name}")
                continue

            print(f"Loading: {mat_file.name}")
            runData = loadmat(mat_file, squeeze_me=True, struct_as_record=False)['runData']
            all_data_dict[session][tag].append(runData)

    print("="*70)
    print("Data loading complete!\n")

    # Print summary
    for session in session_list:
        for tag in tag_list:
            n_runs = len(all_data_dict[session][tag])
            if n_runs > 0:
                print(f"{session} - {tag}: {n_runs} runs loaded")

    return all_data_dict


# ==================== PREPROCESSING FUNCTIONS ====================

def preprocess_eeg_data(all_data_dict, valid_channels=None):
    """
    Preprocess and organize EEG data for CSP

    Parameters:
    -----------
    all_data_dict : dict
        Dictionary containing session and run data
    valid_channels : list
        List of valid channel names to keep

    Returns:
    --------
    processed_data : dict
        Organized trials by session and task
    """
    if valid_channels is None:
        valid_channels = ['F3','F4','Fc5','Fc3','Fc1','Fcz','Fc2','Fc4','Fc6',
                         'T7','C5','C3','C1','Cz','C2','C4','C6','T8','Cp5',
                         'Cp3','Cp1','Cpz','Cp2','Cp4','Cp6','P3','P4']

    processed_data = {}

    for session, tasks in all_data_dict.items():
        processed_data[session] = {}

        for task, runs in tasks.items():
            trials_class1 = []  # Right (LR) or Up (UD)
            trials_class2 = []  # Left (LR) or Down (UD)

            for runData in runs:
                # Get channel indices for valid channels
                labels = runData.label if hasattr(runData, 'label') else []
                if isinstance(labels, np.ndarray):
                    labels = labels.tolist()

                valid_idx = [i for i, label in enumerate(labels)
                           if label in valid_channels]

                fs = runData.fs

                # Process each trial
                for i, trial in enumerate(runData.trials):
                    if runData.outcome[i] == 1:  # Only successful trials
                        # Select valid channels
                        trial_data = trial[valid_idx, :]

                        # Separate by target class
                        if runData.target[i] == 1:
                            trials_class1.append(trial_data)
                        elif runData.target[i] == 2:
                            trials_class2.append(trial_data)

            processed_data[session][task] = {
                'class1': trials_class1,
                'class2': trials_class2,
                'fs': fs
            }

    return processed_data


def apply_csp_pipeline(processed_data, filter_band=(8, 30), n_components=6):
    """
    Apply CSP to preprocessed data

    Parameters:
    -----------
    processed_data : dict
        Output from preprocess_eeg_data
    filter_band : tuple
        Frequency band for filtering (low, high) Hz
    n_components : int
        Number of CSP components

    Returns:
    --------
    csp_features : dict
        CSP features for each session and task
    csp_models : dict
        Fitted CSP models
    """
    csp_features = {}
    csp_models = {}

    for session, tasks in processed_data.items():
        csp_features[session] = {}
        csp_models[session] = {}

        for task, data in tasks.items():
            print(f"\nProcessing {session} - {task}")
            print(f"Class 1 trials: {len(data['class1'])}")
            print(f"Class 2 trials: {len(data['class2'])}")

            # Initialize CSP
            csp = CSPProcessor(n_components=n_components, filter_band=filter_band)

            # Apply bandpass filter to all trials
            fs = data['fs']
            trials_c1_filtered = [csp.bandpass_filter(t, fs) for t in data['class1']]
            trials_c2_filtered = [csp.bandpass_filter(t, fs) for t in data['class2']]

            # Fit and transform
            features_c1, features_c2 = csp.fit_transform(trials_c1_filtered,
                                                         trials_c2_filtered)

            # Store results
            csp_features[session][task] = {
                'features': np.vstack([features_c1, features_c2]),
                'labels': np.hstack([np.ones(len(features_c1)),
                                    np.zeros(len(features_c2))])
            }
            csp_models[session][task] = csp

            print(f"Features shape: {csp_features[session][task]['features'].shape}")

    return csp_features, csp_models


def create_train_test_split(processed_data, test_size=0.2, random_state=42):
    """
    Create train/test splits for each session and task

    Parameters:
    -----------
    processed_data : dict
        Output from preprocess_eeg_data
    test_size : float
        Proportion of data for testing
    random_state : int
        Random seed for reproducibility

    Returns:
    --------
    train_data, test_data : dicts
        Separate dictionaries for train and test data
    """
    np.random.seed(random_state)
    train_data = {}
    test_data = {}

    for session, tasks in processed_data.items():
        train_data[session] = {}
        test_data[session] = {}

        for task, data in tasks.items():
            n_class1 = len(data['class1'])
            n_class2 = len(data['class2'])

            # Create indices for splitting
            idx_c1 = np.random.permutation(n_class1)
            idx_c2 = np.random.permutation(n_class2)

            split_c1 = int(n_class1 * (1 - test_size))
            split_c2 = int(n_class2 * (1 - test_size))

            # Split class 1
            train_c1 = [data['class1'][i] for i in idx_c1[:split_c1]]
            test_c1 = [data['class1'][i] for i in idx_c1[split_c1:]]

            # Split class 2
            train_c2 = [data['class2'][i] for i in idx_c2[:split_c2]]
            test_c2 = [data['class2'][i] for i in idx_c2[split_c2:]]

            train_data[session][task] = {
                'class1': train_c1,
                'class2': train_c2,
                'fs': data['fs']
            }

            test_data[session][task] = {
                'class1': test_c1,
                'class2': test_c2,
                'fs': data['fs']
            }

            print(f"{session}-{task}: Train C1={len(train_c1)}, C2={len(train_c2)} | "
                  f"Test C1={len(test_c1)}, C2={len(test_c2)}")

    return train_data, test_data


def apply_csp_with_split(train_data, test_data, filter_band=(8, 30), n_components=6):
    """
    Apply CSP to train data and transform test data

    Returns:
    --------
    train_features, test_features : dicts
        CSP features for train and test sets
    csp_models : dict
        Fitted CSP models
    """
    train_features = {}
    test_features = {}
    csp_models = {}

    for session in train_data.keys():
        train_features[session] = {}
        test_features[session] = {}
        csp_models[session] = {}

        for task in train_data[session].keys():
            print(f"\nProcessing {session} - {task}")

            # Initialize CSP
            csp = CSPProcessor(n_components=n_components, filter_band=filter_band)

            # Get training data
            train_d = train_data[session][task]
            test_d = test_data[session][task]
            fs = train_d['fs']

            # Apply bandpass filter to training trials
            train_c1_filt = [csp.bandpass_filter(t, fs) for t in train_d['class1']]
            train_c2_filt = [csp.bandpass_filter(t, fs) for t in train_d['class2']]

            # Fit CSP on training data
            feat_c1_train, feat_c2_train = csp.fit_transform(train_c1_filt, train_c2_filt)

            # Apply bandpass filter to test trials
            test_c1_filt = [csp.bandpass_filter(t, fs) for t in test_d['class1']]
            test_c2_filt = [csp.bandpass_filter(t, fs) for t in test_d['class2']]

            # Transform test data using fitted CSP
            feat_c1_test = np.array([csp.transform(t) for t in test_c1_filt])
            feat_c2_test = np.array([csp.transform(t) for t in test_c2_filt])

            # Store results
            train_features[session][task] = {
                'features': np.vstack([feat_c1_train, feat_c2_train]),
                'labels': np.hstack([np.ones(len(feat_c1_train)),
                                    np.zeros(len(feat_c2_train))])
            }

            test_features[session][task] = {
                'features': np.vstack([feat_c1_test, feat_c2_test]),
                'labels': np.hstack([np.ones(len(feat_c1_test)),
                                    np.zeros(len(feat_c2_test))])
            }

            csp_models[session][task] = csp

            print(f"Train features: {train_features[session][task]['features'].shape}")
            print(f"Test features: {test_features[session][task]['features'].shape}")

    return train_features, test_features, csp_models


# ===== CNN Model for CSP Features =====
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class CSPDataset(Dataset):
    """PyTorch Dataset for CSP features"""
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


class CNNModel_CSP(nn.Module):
    """
    CNN model adapted for CSP features
    Input: (batch_size, n_components, sequence_length)
    Output: (batch_size, 2) for binary classification
    """
    def __init__(self, n_components=6, sequence_length=128, n_classes=2):
        super().__init__()

        self.network = nn.Sequential(
            # ----- CNN Layer 1 -----
            nn.Conv1d(in_channels=n_components, out_channels=64,
                     kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.MaxPool1d(kernel_size=2),   # L: 128 -> 64

            # ----- CNN Layer 2 -----
            nn.Conv1d(in_channels=64, out_channels=128,
                     kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.MaxPool1d(kernel_size=2),   # L: 64 -> 32

            # ----- CNN Layer 3 -----
            nn.Conv1d(in_channels=128, out_channels=256,
                     kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.MaxPool1d(kernel_size=2),   # L: 32 -> 16

            nn.Flatten(),
            # Calculate flattened size: 256 channels * 16 time points
            nn.Linear(256 * 16, 120),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(120, n_classes)
        )

    def forward(self, xb):
        return self.network(xb)


def expand_csp_features(features, target_length=128):
    """
    Expand CSP features to sequence for CNN

    Parameters:
    -----------
    features : array (n_samples, n_components)
    target_length : int
        Desired sequence length

    Returns:
    --------
    expanded : array (n_samples, n_components, target_length)
    """
    n_samples, n_components = features.shape
    expanded = np.zeros((n_samples, n_components, target_length))

    # Replicate features across time dimension
    for i in range(n_samples):
        for j in range(n_components):
            expanded[i, j, :] = features[i, j]

    return expanded


def train_cnn_model(train_features, test_features, session, task,
                   n_epochs=50, batch_size=32, lr=0.001, device='cuda'):
    """
    Train CNN model on CSP features

    Returns:
    --------
    model : trained model
    history : training history
    """
    # Prepare data
    X_train = expand_csp_features(train_features[session][task]['features'])
    y_train = train_features[session][task]['labels'].astype(int)

    X_test = expand_csp_features(test_features[session][task]['features'])
    y_test = test_features[session][task]['labels'].astype(int)

    # Create datasets
    train_dataset = CSPDataset(X_train, y_train)
    test_dataset = CSPDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Initialize model
    n_components = X_train.shape[1]
    model = CNNModel_CSP(n_components=n_components, n_classes=2)
    model = model.to(device)

    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Training loop
    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

    for epoch in range(n_epochs):
        # Training
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        # Testing
        model.eval()
        test_loss = 0
        test_correct = 0
        test_total = 0

        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                test_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                test_total += labels.size(0)
                test_correct += (predicted == labels).sum().item()

        # Record history
        history['train_loss'].append(train_loss / len(train_loader))
        history['train_acc'].append(100 * train_correct / train_total)
        history['test_loss'].append(test_loss / len(test_loader))
        history['test_acc'].append(100 * test_correct / test_total)

        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{n_epochs}] "
                  f"Train Loss: {history['train_loss'][-1]:.4f}, "
                  f"Train Acc: {history['train_acc'][-1]:.2f}%, "
                  f"Test Loss: {history['test_loss'][-1]:.4f}, "
                  f"Test Acc: {history['test_acc'][-1]:.2f}%")

    return model, history


# ==================== VISUALIZATION FUNCTIONS ====================

import seaborn as sns
from mpl_toolkits.axes_grid1 import make_axes_locatable
from sklearn.metrics import confusion_matrix

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10


def plot_raw_eeg_sample(runData, trial_idx=0, duration=2.0):
    """Visualize raw EEG from a single trial"""
    trial = runData.trials[trial_idx]
    fs = runData.fs

    # Use actual trial length or requested duration, whichever is smaller
    actual_duration = trial.shape[1] / fs
    duration = min(duration, actual_duration)
    n_samples = int(duration * fs)
    time = np.arange(n_samples) / fs

    # Select subset of channels for clarity
    channels_to_plot = [0, 5, 10, 15, 20, 25]
    channel_labels = [runData.label[i] for i in channels_to_plot]

    fig, axes = plt.subplots(len(channels_to_plot), 1, figsize=(14, 10), sharex=True)
    fig.suptitle(f'Raw EEG - Trial {trial_idx} (Target: {runData.target[trial_idx]}, Duration: {duration:.2f}s)',
                 fontsize=16, fontweight='bold')

    for i, (ch_idx, ax) in enumerate(zip(channels_to_plot, axes)):
        ax.plot(time, trial[ch_idx, :n_samples], linewidth=0.8, color='steelblue')
        ax.set_ylabel(f'{channel_labels[i]}\n(µV)', fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.axvspan(0, min(0.5, duration), alpha=0.1, color='green', label='Baseline' if i == 0 else '')
        if duration > 0.5:
            ax.axvspan(0.5, duration, alpha=0.1, color='orange', label='Motor Imagery' if i == 0 else '')

    axes[-1].set_xlabel('Time (seconds)', fontsize=12)
    axes[0].legend(loc='upper right')
    plt.tight_layout()
    plt.show()
    return fig


def plot_frequency_spectrum(runData, trial_indices=[0, 1], show_bands=True):
    """Show frequency content of EEG signals"""
    from scipy import signal as sig
    fs = runData.fs

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle('Frequency Spectrum Analysis', fontsize=16, fontweight='bold')

    for idx, trial_idx in enumerate(trial_indices):
        trial = runData.trials[trial_idx]
        target = runData.target[trial_idx]

        freqs, psd = sig.welch(trial[13, :], fs, nperseg=512)

        axes[idx].semilogy(freqs, psd, linewidth=2, color='navy', alpha=0.7)
        axes[idx].set_xlabel('Frequency (Hz)', fontsize=11)
        axes[idx].set_ylabel('Power Spectral Density (µV²/Hz)', fontsize=11)
        axes[idx].set_title(f'Trial {trial_idx} - Target: {"Right/Up" if target == 1 else "Left/Down"}',
                           fontsize=12)
        axes[idx].set_xlim([0, 50])
        axes[idx].grid(True, alpha=0.3)

        if show_bands:
            axes[idx].axvspan(8, 13, alpha=0.2, color='green', label='Mu (8-13 Hz)')
            axes[idx].axvspan(13, 30, alpha=0.2, color='orange', label='Beta (13-30 Hz)')
            axes[idx].legend(loc='upper right')

    plt.tight_layout()
    plt.show()
    return fig


def plot_filtering_effect(trial, fs, filter_band=(8, 30), channel_idx=None):
    """Show effect of bandpass filtering"""
    # If no channel specified, use the middle channel
    if channel_idx is None:
        channel_idx = trial.shape[0] // 2

    # Make sure channel_idx is valid
    if channel_idx >= trial.shape[0]:
        channel_idx = trial.shape[0] // 2
        print(f"Warning: Channel index adjusted to {channel_idx} (middle channel)")

    nyq = 0.5 * fs
    low = filter_band[0] / nyq
    high = filter_band[1] / nyq
    sos = butter(5, [low, high], btype='band', output='sos')
    filtered = sosfiltfilt(sos, trial[channel_idx, :])

    time = np.arange(len(trial[channel_idx, :])) / fs

    # Use actual trial length or 2 seconds, whichever is smaller
    actual_duration = len(trial[channel_idx, :]) / fs
    duration = min(2.0, actual_duration)
    n_samples = int(duration * fs)

    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    fig.suptitle(f'Bandpass Filtering Effect ({filter_band[0]}-{filter_band[1]} Hz) - Channel {channel_idx}',
                 fontsize=16, fontweight='bold')

    axes[0].plot(time[:n_samples], trial[channel_idx, :n_samples],
                linewidth=0.8, color='steelblue')
    axes[0].set_ylabel('Raw EEG (µV)', fontsize=11)
    axes[0].set_title('Original Signal', fontsize=12)
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(time[:n_samples], filtered[:n_samples],
                linewidth=0.8, color='darkgreen')
    axes[1].set_ylabel('Filtered EEG (µV)', fontsize=11)
    axes[1].set_title('After Bandpass Filter (Mu + Beta bands)', fontsize=12)
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(time[:n_samples], trial[channel_idx, :n_samples],
                linewidth=0.8, color='steelblue', alpha=0.5, label='Original')
    axes[2].plot(time[:n_samples], filtered[:n_samples],
                linewidth=0.8, color='darkgreen', label='Filtered')
    axes[2].set_xlabel('Time (seconds)', fontsize=11)
    axes[2].set_ylabel('Amplitude (µV)', fontsize=11)
    axes[2].set_title('Comparison', fontsize=12)
    axes[2].legend(loc='upper right')
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return fig


def plot_csp_patterns(csp_model, channel_labels=None):
    """Visualize CSP spatial filters"""
    filters = csp_model.filters
    n_channels = filters.shape[1]

    # If no labels provided or mismatch, create generic labels
    if channel_labels is None or len(channel_labels) != n_channels:
        channel_labels = [f'Ch{i+1}' for i in range(n_channels)]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(f'CSP Spatial Patterns (Top 3 for each class) - {n_channels} channels',
                 fontsize=16, fontweight='bold')

    axes = axes.flatten()

    for i in range(6):
        pattern = filters[i, :]
        colors = ['red' if p < 0 else 'blue' for p in pattern]
        axes[i].bar(range(len(pattern)), pattern, color=colors, alpha=0.7)
        axes[i].set_xticks(range(len(pattern)))
        axes[i].set_xticklabels(channel_labels, rotation=90, fontsize=8)
        axes[i].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
        axes[i].set_ylabel('Weight', fontsize=10)
        axes[i].grid(True, alpha=0.3, axis='y')

        if i < 3:
            axes[i].set_title(f'CSP Component {i+1} (Class 1)', fontsize=11, color='darkred')
        else:
            axes[i].set_title(f'CSP Component {i+1} (Class 2)', fontsize=11, color='darkblue')

    plt.tight_layout()
    plt.show()
    return fig


def plot_eigenvalue_spectrum(csp_model):
    """Plot CSP eigenvalue spectrum"""
    eigenvalues = csp_model.eigenvalues

    fig, ax = plt.subplots(1, 1, figsize=(12, 6))

    ax.plot(range(len(eigenvalues)), eigenvalues, 'o-',
           linewidth=2, markersize=8, color='navy')
    ax.axhline(y=0.5, color='red', linestyle='--', linewidth=2,
              label='Equal variance (0.5)')

    n_comp = 3
    ax.scatter(range(n_comp), eigenvalues[:n_comp],
              s=200, c='red', marker='o', edgecolors='black',
              linewidth=2, label='Selected (Class 1)', zorder=5)
    ax.scatter(range(len(eigenvalues)-n_comp, len(eigenvalues)),
              eigenvalues[-n_comp:],
              s=200, c='blue', marker='o', edgecolors='black',
              linewidth=2, label='Selected (Class 2)', zorder=5)

    ax.set_xlabel('Component Index', fontsize=12)
    ax.set_ylabel('Eigenvalue (Variance Ratio)', fontsize=12)
    ax.set_title('CSP Eigenvalue Spectrum\n(Higher = More discriminative for Class 1)',
                fontsize=14, fontweight='bold')
    ax.legend(loc='right', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([-0.1, 1.1])

    plt.tight_layout()
    plt.show()
    return fig


def plot_csp_features_distribution(train_features, session='Session1', task='LR'):
    """Visualize distribution of CSP features for both classes"""
    features = train_features[session][task]['features']
    labels = train_features[session][task]['labels']

    class1_features = features[labels == 1]
    class2_features = features[labels == 0]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(f'CSP Feature Distributions - {session} {task}',
                 fontsize=16, fontweight='bold')

    axes = axes.flatten()

    for i in range(6):
        axes[i].hist(class1_features[:, i], bins=20, alpha=0.6,
                    color='red', label='Class 1 (Right/Up)', density=True)
        axes[i].hist(class2_features[:, i], bins=20, alpha=0.6,
                    color='blue', label='Class 2 (Left/Down)', density=True)
        axes[i].set_xlabel(f'CSP Feature {i+1} (log variance)', fontsize=10)
        axes[i].set_ylabel('Density', fontsize=10)
        axes[i].set_title(f'Component {i+1}', fontsize=11)
        axes[i].legend(loc='upper right', fontsize=9)
        axes[i].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return fig


def plot_csp_features_2d(train_features, session='Session1', task='LR'):
    """2D scatter plot of most discriminative CSP features"""
    features = train_features[session][task]['features']
    labels = train_features[session][task]['labels']

    class1_features = features[labels == 1]
    class2_features = features[labels == 0]

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle(f'CSP Feature Space - {session} {task}',
                 fontsize=16, fontweight='bold')

    axes[0].scatter(class1_features[:, 0], class1_features[:, 1],
                   c='red', alpha=0.6, s=50, label='Class 1 (Right/Up)', edgecolors='black')
    axes[0].scatter(class2_features[:, 0], class2_features[:, 1],
                   c='blue', alpha=0.6, s=50, label='Class 2 (Left/Down)', edgecolors='black')
    axes[0].set_xlabel('CSP Component 1', fontsize=12)
    axes[0].set_ylabel('CSP Component 2', fontsize=12)
    axes[0].set_title('Components 1 vs 2', fontsize=12)
    axes[0].legend(loc='best')
    axes[0].grid(True, alpha=0.3)

    axes[1].scatter(class1_features[:, -2], class1_features[:, -1],
                   c='red', alpha=0.6, s=50, label='Class 1 (Right/Up)', edgecolors='black')
    axes[1].scatter(class2_features[:, -2], class2_features[:, -1],
                   c='blue', alpha=0.6, s=50, label='Class 2 (Left/Down)', edgecolors='black')
    axes[1].set_xlabel('CSP Component 5', fontsize=12)
    axes[1].set_ylabel('CSP Component 6', fontsize=12)
    axes[1].set_title('Components 5 vs 6', fontsize=12)
    axes[1].legend(loc='best')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return fig


def plot_feature_expansion():
    """Illustrate feature expansion process"""
    csp_features = np.array([2.5, 1.8, 2.1, -0.3, -0.5, -0.2])
    expanded = np.tile(csp_features.reshape(-1, 1), (1, 128))

    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    fig.suptitle('Feature Expansion for CNN Input', fontsize=16, fontweight='bold')

    axes[0].bar(range(6), csp_features, color=['red', 'red', 'red', 'blue', 'blue', 'blue'],
               alpha=0.7, edgecolor='black')
    axes[0].set_xlabel('CSP Component', fontsize=12)
    axes[0].set_ylabel('Log Variance', fontsize=12)
    axes[0].set_title('Original CSP Features (6 values)', fontsize=12)
    axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
    axes[0].grid(True, alpha=0.3, axis='y')
    axes[0].set_xticks(range(6))
    axes[0].set_xticklabels([f'CSP-{i+1}' for i in range(6)])

    im = axes[1].imshow(expanded, aspect='auto', cmap='RdBu_r',
                       interpolation='nearest', vmin=-3, vmax=3)
    axes[1].set_xlabel('Time Steps (Replicated)', fontsize=12)
    axes[1].set_ylabel('CSP Component', fontsize=12)
    axes[1].set_title('Expanded Features (6 × 128) - Ready for CNN', fontsize=12)
    axes[1].set_yticks(range(6))
    axes[1].set_yticklabels([f'CSP-{i+1}' for i in range(6)])

    divider = make_axes_locatable(axes[1])
    cax = divider.append_axes("right", size="2%", pad=0.1)
    plt.colorbar(im, cax=cax, label='Log Variance')

    plt.tight_layout()
    plt.show()
    return fig


def plot_training_history(history, session='Session1', task='LR'):
    """Plot training and validation metrics over epochs"""
    epochs = range(1, len(history['train_loss']) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle(f'Training History - {session} {task}', fontsize=16, fontweight='bold')

    axes[0].plot(epochs, history['train_loss'], 'o-',
                linewidth=2, markersize=4, color='blue', label='Train Loss')
    axes[0].plot(epochs, history['test_loss'], 's-',
                linewidth=2, markersize=4, color='red', label='Test Loss')
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title('Loss vs Epoch', fontsize=13)
    axes[0].legend(loc='upper right', fontsize=11)
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history['train_acc'], 'o-',
                linewidth=2, markersize=4, color='blue', label='Train Accuracy')
    axes[1].plot(epochs, history['test_acc'], 's-',
                linewidth=2, markersize=4, color='red', label='Test Accuracy')
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Accuracy (%)', fontsize=12)
    axes[1].set_title('Accuracy vs Epoch', fontsize=13)
    axes[1].legend(loc='lower right', fontsize=11)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim([0, 105])

    final_train_acc = history['train_acc'][-1]
    final_test_acc = history['test_acc'][-1]
    axes[1].text(0.02, 0.98,
                f'Final Train: {final_train_acc:.2f}%\nFinal Test: {final_test_acc:.2f}%',
                transform=axes[1].transAxes, fontsize=11, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.tight_layout()
    plt.show()
    return fig


def plot_confusion_matrix_bci(model, test_features, session='Session1', task='LR', device='cuda'):
    """Plot confusion matrix for test set predictions"""
    X_test = expand_csp_features(test_features[session][task]['features'])
    y_test = test_features[session][task]['labels'].astype(int)

    model.eval()
    X_test_tensor = torch.FloatTensor(X_test).to(device)

    with torch.no_grad():
        outputs = model(X_test_tensor)
        _, predictions = torch.max(outputs, 1)

    predictions = predictions.cpu().numpy()
    cm = confusion_matrix(y_test, predictions)

    fig, ax = plt.subplots(1, 1, figsize=(8, 6))

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Left/Down', 'Right/Up'],
                yticklabels=['Left/Down', 'Right/Up'],
                cbar_kws={'label': 'Count'}, ax=ax)

    ax.set_xlabel('Predicted Label', fontsize=12)
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_title(f'Confusion Matrix - {session} {task}\nTest Accuracy: {100 * cm.diagonal().sum() / cm.sum():.2f}%',
                fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.show()
    return fig, cm


def calculate_pvc(all_data_dict, session, task):
    """
    Calculate Peak Validation Criterion (PVC) from online classifier performance

    PVC = (# hit trials / # non-aborted trials) * 100

    Parameters:
    -----------
    all_data_dict : dict
        Dictionary containing session and run data
    session : str
        Session name (e.g., 'Session1')
    task : str
        Task name (e.g., 'LR' or 'UD')

    Returns:
    --------
    pvc_score : float
        PVC percentage
    hit_count : int
        Number of hit trials
    miss_count : int
        Number of miss trials
    abort_count : int
        Number of aborted trials
    """
    hit_count = 0
    miss_count = 0
    abort_count = 0

    for runData in all_data_dict[session][task]:
        for outcome in runData.outcome:
            if outcome == 1:  # Hit
                hit_count += 1
            elif outcome == -1:  # Miss
                miss_count += 1
            elif outcome == 0:  # Abort
                abort_count += 1

    non_aborted = hit_count + miss_count
    pvc_score = (hit_count / non_aborted * 100) if non_aborted > 0 else 0

    return pvc_score, hit_count, miss_count, abort_count


def plot_pvc_comparison(all_data_dict, all_results):
    """
    Compare PVC (online classifier) vs offline CNN accuracy

    Parameters:
    -----------
    all_data_dict : dict
        Original data with online classifier outcomes
    all_results : dict
        CNN training results with test accuracies
    """
    data = []

    for session in all_results.keys():
        for task in all_results[session].keys():
            # Calculate PVC from online data
            pvc_score, hits, misses, aborts = calculate_pvc(all_data_dict, session, task)

            # Get offline CNN accuracy
            cnn_accuracy = all_results[session][task]['test_acc'][-1]

            data.append({
                'session': session,
                'task': task,
                'pvc': pvc_score,
                'cnn_acc': cnn_accuracy,
                'hits': hits,
                'misses': misses,
                'aborts': aborts
            })

    # Create comparison plot
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Online Linear Classifier (PVC) vs Offline CNN Performance',
                 fontsize=16, fontweight='bold')

    # Plot 1: Bar chart comparison
    x = np.arange(len(data))
    width = 0.35

    pvc_scores = [d['pvc'] for d in data]
    cnn_scores = [d['cnn_acc'] for d in data]
    labels = [f"{d['session']}\n{d['task']}" for d in data]

    bars1 = axes[0].bar(x - width/2, pvc_scores, width,
                       label='Online PVC', color='coral', alpha=0.8, edgecolor='black')
    bars2 = axes[0].bar(x + width/2, cnn_scores, width,
                       label='Offline CNN', color='steelblue', alpha=0.8, edgecolor='black')

    axes[0].set_xlabel('Session - Task', fontsize=12)
    axes[0].set_ylabel('Accuracy (%)', fontsize=12)
    axes[0].set_title('PVC vs CNN Accuracy Comparison', fontsize=13)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels)
    axes[0].set_ylim([0, 105])
    axes[0].axhline(y=50, color='red', linestyle='--', linewidth=2, label='Chance Level (50%)')
    axes[0].legend(loc='lower right', fontsize=11)
    axes[0].grid(True, alpha=0.3, axis='y')

    # Add value labels on bars
    for bar in bars1:
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height + 1,
                    f'{height:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
    for bar in bars2:
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height + 1,
                    f'{height:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

    # Plot 2: Scatter plot (PVC vs CNN)
    colors_map = {'LR': 'steelblue', 'UD': 'coral'}
    for d in data:
        color = colors_map[d['task']]
        axes[1].scatter(d['pvc'], d['cnn_acc'], s=200, c=color, alpha=0.7,
                       edgecolors='black', linewidth=2, label=d['task'] if d['session'] == 'Session1' else '')
        axes[1].text(d['pvc'] + 1, d['cnn_acc'] + 1,
                    f"{d['session']}\n{d['task']}", fontsize=9, ha='left')

    # Add diagonal line (perfect correlation)
    min_val = min(min(pvc_scores), min(cnn_scores)) - 5
    max_val = max(max(pvc_scores), max(cnn_scores)) + 5
    axes[1].plot([min_val, max_val], [min_val, max_val],
                'k--', linewidth=2, alpha=0.5, label='Perfect Correlation')

    axes[1].set_xlabel('Online PVC (%)', fontsize=12)
    axes[1].set_ylabel('Offline CNN Accuracy (%)', fontsize=12)
    axes[1].set_title('Correlation: Online vs Offline Performance', fontsize=13)
    axes[1].legend(loc='lower right', fontsize=10)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim([min_val, max_val])
    axes[1].set_ylim([min_val, max_val])

    plt.tight_layout()
    plt.show()

    return fig, data


def plot_trial_outcomes_breakdown(all_data_dict):
    """
    Visualize the breakdown of trial outcomes (hit/miss/abort) for each session-task
    """
    data = []

    for session in all_data_dict.keys():
        for task in all_data_dict[session].keys():
            pvc_score, hits, misses, aborts = calculate_pvc(all_data_dict, session, task)
            total = hits + misses + aborts

            data.append({
                'session': session,
                'task': task,
                'hits': hits,
                'misses': misses,
                'aborts': aborts,
                'total': total,
                'pvc': pvc_score
            })

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Online Classifier Trial Outcomes Breakdown', fontsize=16, fontweight='bold')

    # Plot 1: Stacked bar chart
    x = np.arange(len(data))
    labels = [f"{d['session']}\n{d['task']}" for d in data]

    hits = [d['hits'] for d in data]
    misses = [d['misses'] for d in data]
    aborts = [d['aborts'] for d in data]

    axes[0].bar(x, hits, label='Hits', color='green', alpha=0.7, edgecolor='black')
    axes[0].bar(x, misses, bottom=hits, label='Misses', color='red', alpha=0.7, edgecolor='black')
    axes[0].bar(x, aborts, bottom=[h+m for h, m in zip(hits, misses)],
               label='Aborts (ignored in PVC)', color='gray', alpha=0.7, edgecolor='black')

    axes[0].set_xlabel('Session - Task', fontsize=12)
    axes[0].set_ylabel('Number of Trials', fontsize=12)
    axes[0].set_title('Trial Outcomes Distribution', fontsize=13)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels)
    axes[0].legend(loc='upper right', fontsize=11)
    axes[0].grid(True, alpha=0.3, axis='y')

    # Add total count on top
    for i, d in enumerate(data):
        axes[0].text(i, d['total'] + 2, f"n={d['total']}",
                    ha='center', va='bottom', fontsize=10, fontweight='bold')

    # Plot 2: PVC with trial breakdown
    x = np.arange(len(data))
    colors = ['steelblue' if d['task'] == 'LR' else 'coral' for d in data]
    pvcs = [d['pvc'] for d in data]

    bars = axes[1].bar(x, pvcs, color=colors, alpha=0.8, edgecolor='black')

    axes[1].set_xlabel('Session - Task', fontsize=12)
    axes[1].set_ylabel('PVC (%)', fontsize=12)
    axes[1].set_title('Percent Valid Correct (PVC)\nHits / (Hits + Misses)', fontsize=13)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(labels)
    axes[1].set_ylim([0, 105])
    axes[1].axhline(y=50, color='red', linestyle='--', linewidth=2, label='Chance Level (50%)')
    axes[1].legend(loc='lower right')
    axes[1].grid(True, alpha=0.3, axis='y')

    # Add detailed labels
    for i, (bar, d) in enumerate(zip(bars, data)):
        height = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2., height + 1,
                    f'{height:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
        axes[1].text(bar.get_x() + bar.get_width()/2., height/2,
                    f'{d["hits"]}/{d["hits"]+d["misses"]}',
                    ha='center', va='center', fontsize=9, color='white', fontweight='bold')

    plt.tight_layout()
    plt.show()

    return fig, data

def plot_pipeline_summary(all_results):
    """Summary of results across all sessions and tasks"""
    sessions = list(all_results.keys())
    tasks = ['LR', 'UD']

    data = []
    for session in sessions:
        for task in tasks:
            if task in all_results[session]:
                data.append({
                    'session': session,
                    'task': task,
                    'accuracy': all_results[session][task]['test_acc'][-1]
                })

    fig, ax = plt.subplots(1, 1, figsize=(10, 6))

    x = np.arange(len(data))
    colors = ['steelblue' if d['task'] == 'LR' else 'coral' for d in data]
    bars = ax.bar(x, [d['accuracy'] for d in data], color=colors, alpha=0.8, edgecolor='black')

    ax.set_xlabel('Session - Task', fontsize=12)
    ax.set_ylabel('Test Accuracy (%)', fontsize=12)
    ax.set_title('BCI Classification Performance Summary', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([f"{d['session']}\n{d['task']}" for d in data])
    ax.set_ylim([0, 105])
    ax.axhline(y=50, color='red', linestyle='--', linewidth=2, label='Chance Level (50%)')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3, axis='y')

    for bar, d in zip(bars, data):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 1,
               f'{d["accuracy"]:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.show()
    return fig


def print_pvc_summary(all_data_dict, all_results):
    """Print detailed PVC and CNN accuracy comparison"""
    print("\n" + "="*80)
    print("PVC (ONLINE LINEAR CLASSIFIER) vs CNN (OFFLINE) PERFORMANCE COMPARISON")
    print("="*80)
    print("\nNote: PVC only considers non-aborted trials (hits + misses)")
    print("      This makes online and offline performance directly comparable\n")

    print(f"{'Session':<12} {'Task':<6} {'Hits':<6} {'Miss':<6} {'Abort':<7} {'PVC':<8} {'CNN Acc':<10} {'Improvement':<12}")
    print("-" * 80)

    for session in all_results.keys():
        for task in all_results[session].keys():
            pvc_score, hits, misses, aborts = calculate_pvc(all_data_dict, session, task)
            cnn_acc = all_results[session][task]['test_acc'][-1]
            improvement = cnn_acc - pvc_score

            print(f"{session:<12} {task:<6} {hits:<6} {misses:<6} {aborts:<7} "
                  f"{pvc_score:>6.2f}%  {cnn_acc:>8.2f}%  {improvement:>+10.2f}%")

    print("-" * 80)

    # Calculate averages
    all_pvc = []
    all_cnn = []
    for session in all_results.keys():
        for task in all_results[session].keys():
            pvc_score, _, _, _ = calculate_pvc(all_data_dict, session, task)
            cnn_acc = all_results[session][task]['test_acc'][-1]
            all_pvc.append(pvc_score)
            all_cnn.append(cnn_acc)

    avg_pvc = np.mean(all_pvc)
    avg_cnn = np.mean(all_cnn)
    avg_improvement = avg_cnn - avg_pvc

    print(f"\n{'AVERAGE':<12} {'ALL':<6} {'':<6} {'':<6} {'':<7} "
          f"{avg_pvc:>6.2f}%  {avg_cnn:>8.2f}%  {avg_improvement:>+10.2f}%")
    print("="*80 + "\n")
    """Summary of results across all sessions and tasks"""
    sessions = list(all_results.keys())
    tasks = ['LR', 'UD']

    data = []
    for session in sessions:
        for task in tasks:
            if task in all_results[session]:
                data.append({
                    'session': session,
                    'task': task,
                    'accuracy': all_results[session][task]['test_acc'][-1]
                })

    fig, ax = plt.subplots(1, 1, figsize=(10, 6))

    x = np.arange(len(data))
    colors = ['steelblue' if d['task'] == 'LR' else 'coral' for d in data]
    bars = ax.bar(x, [d['accuracy'] for d in data], color=colors, alpha=0.8, edgecolor='black')

    ax.set_xlabel('Session - Task', fontsize=12)
    ax.set_ylabel('Test Accuracy (%)', fontsize=12)
    ax.set_title('BCI Classification Performance Summary', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([f"{d['session']}\n{d['task']}" for d in data])
    ax.set_ylim([0, 105])
    ax.axhline(y=50, color='red', linestyle='--', linewidth=2, label='Chance Level (50%)')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3, axis='y')

    for bar, d in zip(bars, data):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 1,
               f'{d["accuracy"]:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.show()
    return fig


# Example usage:
if __name__ == "__main__":
    # ===== LOAD DATA FIRST =====
    print("\n" + "="*70)
    print("LOADING BCI DATA")
    print("="*70)

    # IMPORTANT: Set your data folder path here
    DATA_FOLDER = "/content/drive/MyDrive/Group1 BCI SMR Data"
    # Or use the variable if already defined
    try:
        # Check if all_data_dict already exists
        if 'all_data_dict' not in locals() and 'all_data_dict' not in globals():
            # If not, try to load it
            print("all_data_dict not found. Please provide the data folder path.")
            print("Example: DATA_FOLDER = '/content/drive/MyDrive/Group1 BCI SMR Data'")
            print("\nYou can either:")
            print("1. Run your data loading code first, OR")
            print("2. Uncomment and set DATA_FOLDER below:\n")
            print("# DATA_FOLDER = Path(r'/your/path/here')")
            print("# all_data_dict = load_bci_data(DATA_FOLDER)")
            raise NameError("all_data_dict not defined. Please load data first.")
    except NameError:
        # Try loading if DATA_FOLDER is defined
        if 'DATA_FOLDER' in locals() or 'DATA_FOLDER' in globals():
            all_data_dict = load_bci_data(DATA_FOLDER)
        else:
            raise NameError("Please define DATA_FOLDER or load all_data_dict before running")

    # Step 1: Preprocess and split data
    print("\n" + "="*70)
    print("STEP 1: PREPROCESSING AND SPLITTING DATA")
    print("="*70)
    processed_data = preprocess_eeg_data(all_data_dict)
    train_data, test_data = create_train_test_split(processed_data, test_size=0.2)

    # Step 2: Apply CSP with train/test split
    print("\n" + "="*70)
    print("STEP 2: APPLYING CSP TO TRAIN AND TEST DATA")
    print("="*70)
    train_features, test_features, csp_models = apply_csp_with_split(
        train_data, test_data,
        filter_band=(8, 30),
        n_components=6
    )

    # ===== VISUALIZATION SECTION =====
    print("\n" + "="*70)
    print("VISUALIZATION SECTION: GENERATING ALL PLOTS")
    print("="*70)

    # Define valid channels
    valid_channels = ['F3','F4','Fc5','Fc3','Fc1','Fcz','Fc2','Fc4','Fc6',
                      'T7','C5','C3','C1','Cz','C2','C4','C6','T8','Cp5',
                      'Cp3','Cp1','Cpz','Cp2','Cp4','Cp6','P3','P4']

    # Generate visualizations for both sessions and both tasks
    for session in ['Session1', 'Session2']:
        for task in ['LR', 'UD']:
            print(f"\n{'='*70}")
            print(f"VISUALIZATIONS FOR {session} - {task}")
            print(f"{'='*70}")

            # Check if this session-task combination exists
            if session not in all_data_dict or task not in all_data_dict[session]:
                print(f"Warning: {session}-{task} not found in data. Skipping...")
                continue

            if len(all_data_dict[session][task]) == 0:
                print(f"Warning: {session}-{task} has no data. Skipping...")
                continue

            # Visualization 1: Raw EEG sample
            print(f"\n=== 1. Raw EEG Sample - {session} {task} ===")
            runData = all_data_dict[session][task][0]
            plot_raw_eeg_sample(runData, trial_idx=5, duration=2.0)

            # Visualization 2: Frequency spectrum
            print(f"\n=== 2. Frequency Spectrum - {session} {task} ===")
            plot_frequency_spectrum(runData, trial_indices=[0, 10])

            # Visualization 3: Filtering effect
            print(f"\n=== 3. Bandpass Filtering Effect - {session} {task} ===")
            if session in processed_data and task in processed_data[session]:
                trial = processed_data[session][task]['class1'][0]
                plot_filtering_effect(trial, fs=300, filter_band=(8, 30))

            # Visualization 4: CSP patterns
            print(f"\n=== 4. CSP Spatial Patterns - {session} {task} ===")
            if session in csp_models and task in csp_models[session]:
                first_run = all_data_dict[session][task][0]
                actual_labels = [label for label in first_run.label if label in valid_channels]
                plot_csp_patterns(csp_models[session][task], actual_labels)

            # Visualization 5: Eigenvalue spectrum
            print(f"\n=== 5. CSP Eigenvalue Spectrum - {session} {task} ===")
            if session in csp_models and task in csp_models[session]:
                plot_eigenvalue_spectrum(csp_models[session][task])

            # Visualization 6: Feature distributions
            print(f"\n=== 6. CSP Feature Distributions - {session} {task} ===")
            if session in train_features and task in train_features[session]:
                plot_csp_features_distribution(train_features, session, task)

            # Visualization 7: 2D feature space
            print(f"\n=== 7. 2D Feature Space - {session} {task} ===")
            if session in train_features and task in train_features[session]:
                plot_csp_features_2d(train_features, session, task)

    # Visualization 8: Feature expansion (only once, it's generic)
    print(f"\n{'='*70}")
    print("=== 8. Feature Expansion Illustration (Generic) ===")
    print(f"{'='*70}")
    plot_feature_expansion()

    # ===== TRAINING SECTION =====
    print("\n" + "="*70)
    print("STEP 3: TRAINING CNN MODELS FOR ALL SESSION-TASK COMBINATIONS")
    print("="*70)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    all_results = {}
    all_models = {}

    for session in train_features.keys():
        all_results[session] = {}
        all_models[session] = {}

        for task in train_features[session].keys():
            print(f"\n{'='*70}")
            print(f"TRAINING: {session} - {task}")
            print(f"{'='*70}")

            # Train the model
            model, history = train_cnn_model(
                train_features, test_features,
                session, task,
                n_epochs=50,
                batch_size=32,
                lr=0.001,
                device=device
            )

            all_results[session][task] = history
            all_models[session][task] = model

            print(f"\n✓ Final Test Accuracy: {history['test_acc'][-1]:.2f}%")

            # Visualization 9: Training history
            print(f"\n=== 9. Training History - {session} {task} ===")
            plot_training_history(history, session, task)

            # Visualization 10: Confusion matrix
            print(f"\n=== 10. Confusion Matrix - {session} {task} ===")
            plot_confusion_matrix_bci(model, test_features, session, task, device)

    # Visualization 11: Overall summary
    print(f"\n{'='*70}")
    print("=== 11. Overall Performance Summary - All Sessions & Tasks ===")
    print(f"{'='*70}")
    plot_pipeline_summary(all_results)

    # Visualization 12: PVC comparison
    print(f"\n{'='*70}")
    print("=== 12. PVC vs CNN Performance Comparison ===")
    print(f"{'='*70}")
    plot_pvc_comparison(all_data_dict, all_results)

    # Visualization 13: Trial outcomes breakdown
    print(f"\n{'='*70}")
    print("=== 13. Trial Outcomes Breakdown (Hit/Miss/Abort) ===")
    print(f"{'='*70}")
    plot_trial_outcomes_breakdown(all_data_dict)

    # Print detailed PVC summary
    print_pvc_summary(all_data_dict, all_results)

    # ===== FINAL SUMMARY =====
    print("\n" + "="*70)
    print("FINAL RESULTS SUMMARY")
    print("="*70)
    for session in all_results.keys():
        print(f"\n{session}:")
        for task in all_results[session].keys():
            final_acc = all_results[session][task]['test_acc'][-1]
            print(f"  {task}: {final_acc:.2f}% test accuracy")

    print("\n" + "="*70)
    print("PIPELINE COMPLETED SUCCESSFULLY!")
    print("="*70)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Step 1: Preprocess and split data
print("\n" + "="*70)
print("STEP 1: PREPROCESSING AND SPLITTING DATA")
print("="*70)

# Set include_misses=True to use same trials as PVC (import numpy as np
from scipy.linalg import eigh
from scipy.signal import butter, filtfilt, sosfiltfilt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.io import loadmat

class CSPProcessor:
    """Common Spatial Pattern processor for BCI EEG data"""

    def __init__(self, n_components=6, filter_band=(8, 30)):
        """
        Parameters:
        -----------
        n_components : int
            Number of CSP components to use (3 from each class)
        filter_band : tuple
            Frequency band to filter (low, high) in Hz
        """
        self.n_components = n_components
        self.filter_band = filter_band
        self.filters = None
        self.eigenvalues = None

    def bandpass_filter(self, data, fs, order=5):
        """Apply bandpass filter to EEG data"""
        nyq = 0.5 * fs
        low = self.filter_band[0] / nyq
        high = self.filter_band[1] / nyq
        sos = butter(order, [low, high], btype='band', output='sos')

        # Apply filter to each channel
        filtered_data = np.zeros_like(data)
        for ch in range(data.shape[0]):
            filtered_data[ch, :] = sosfiltfilt(sos, data[ch, :])

        return filtered_data

    def compute_covariance(self, trials):
        """Compute normalized covariance matrix for trials"""
        n_channels = trials[0].shape[0]
        cov = np.zeros((n_channels, n_channels))

        for trial in trials:
            # Normalize by trace
            trial_cov = np.dot(trial, trial.T) / np.trace(np.dot(trial, trial.T))
            cov += trial_cov

        return cov / len(trials)

    def fit(self, trials_class1, trials_class2):
        """
        Fit CSP filters

        Parameters:
        -----------
        trials_class1 : list of arrays
            Each array is (n_channels, n_samples) for class 1
        trials_class2 : list of arrays
            Each array is (n_channels, n_samples) for class 2
        """
        # Compute covariance matrices for each class
        cov1 = self.compute_covariance(trials_class1)
        cov2 = self.compute_covariance(trials_class2)

        # Solve generalized eigenvalue problem
        eigenvalues, eigenvectors = eigh(cov1, cov1 + cov2)

        # Sort by eigenvalues (descending)
        idx = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]

        # Select top and bottom components
        n_comp_per_class = self.n_components // 2
        selected_filters = np.hstack([
            eigenvectors[:, :n_comp_per_class],  # Top components (class 1)
            eigenvectors[:, -n_comp_per_class:]  # Bottom components (class 2)
        ])

        self.filters = selected_filters.T  # Shape: (n_components, n_channels)
        self.eigenvalues = eigenvalues

        return self

    def transform(self, trial):
        """
        Apply CSP transformation to a single trial

        Parameters:
        -----------
        trial : array (n_channels, n_samples)

        Returns:
        --------
        features : array (n_components,)
            Log variance of CSP-filtered signals
        """
        if self.filters is None:
            raise ValueError("CSP filters not fitted. Call fit() first.")

        # Apply spatial filters
        filtered = np.dot(self.filters, trial)  # (n_components, n_samples)

        # Compute log variance for each component
        variances = np.var(filtered, axis=1)
        features = np.log(variances + 1e-8)  # Add small constant for stability

        return features

    def fit_transform(self, trials_class1, trials_class2):
        """Fit and transform training data"""
        self.fit(trials_class1, trials_class2)

        # Transform all trials
        features_class1 = np.array([self.transform(t) for t in trials_class1])
        features_class2 = np.array([self.transform(t) for t in trials_class2])

        return features_class1, features_class2


# ==================== DATA LOADING FUNCTION ====================

def load_bci_data(data_folder_path):
    """
    Load BCI data from .mat files

    Parameters:
    -----------
    data_folder_path : str or Path
        Path to the folder containing Session1 and Session2 folders

    Returns:
    --------
    all_data_dict : dict
        Dictionary with structure: {session: {task: [runData1, runData2, ...]}}
    """
    DATA_FOLDER = Path(data_folder_path)
    session_list = ["Session1", "Session2"]
    tag_list = ["LR", "UD"]

    all_data_dict = {
        session: {tag: [] for tag in tag_list}
        for session in session_list
    }

    print("Loading BCI data files...")
    print("="*70)

    for session in session_list:
        session_path = DATA_FOLDER / session

        if not session_path.exists():
            print(f"Warning: {session} folder not found at {session_path}")
            continue

        for mat_file in session_path.glob("*.mat"):
            tag = mat_file.stem.split("_")[1][:2]  # 'LR' or 'UD'

            if tag not in tag_list:
                print(f"Warning: Unknown tag {tag} in file {mat_file.name}")
                continue

            print(f"Loading: {mat_file.name}")
            runData = loadmat(mat_file, squeeze_me=True, struct_as_record=False)['runData']
            all_data_dict[session][tag].append(runData)

    print("="*70)
    print("Data loading complete!\n")

    # Print summary
    for session in session_list:
        for tag in tag_list:
            n_runs = len(all_data_dict[session][tag])
            if n_runs > 0:
                print(f"{session} - {tag}: {n_runs} runs loaded")

    return all_data_dict


# ==================== PREPROCESSING FUNCTIONS ====================

def preprocess_eeg_data(all_data_dict, valid_channels=None, include_misses=True):
    """
    Preprocess and organize EEG data for CSP

    Parameters:
    -----------
    all_data_dict : dict
        Dictionary containing session and run data
    valid_channels : list
        List of valid channel names to keep
    include_misses : bool
        If True, include both hits and misses (outcome = 1 or -1) - RECOMMENDED for PVC comparison
        If False, include only hits (outcome = 1)

    Returns:
    --------
    processed_data : dict
        Organized trials by session and task
    """
    if valid_channels is None:
        valid_channels = ['F3','F4','Fc5','Fc3','Fc1','Fcz','Fc2','Fc4','Fc6',
                         'T7','C5','C3','C1','Cz','C2','C4','C6','T8','Cp5',
                         'Cp3','Cp1','Cpz','Cp2','Cp4','Cp6','P3','P4']

    processed_data = {}

    print(f"\nPreprocessing with: {'HITS + MISSES' if include_misses else 'HITS ONLY'}")
    print("-" * 70)

    for session, tasks in all_data_dict.items():
        processed_data[session] = {}

        for task, runs in tasks.items():
            trials_class1 = []  # Right (LR) or Up (UD)
            trials_class2 = []  # Left (LR) or Down (UD)

            hit_count = 0
            miss_count = 0
            abort_count = 0

            for runData in runs:
                # Get channel indices for valid channels
                labels = runData.label if hasattr(runData, 'label') else []
                if isinstance(labels, np.ndarray):
                    labels = labels.tolist()

                valid_idx = [i for i, label in enumerate(labels)
                           if label in valid_channels]

                fs = runData.fs

                # Process each trial
                for i, trial in enumerate(runData.trials):
                    outcome = runData.outcome[i]

                    # Count outcomes
                    if outcome == 1:
                        hit_count += 1
                    elif outcome == -1:
                        miss_count += 1
                    elif outcome == 0:
                        abort_count += 1

                    # Decide which trials to include
                    if include_misses:
                        # Include both hits and misses, exclude aborts
                        include_trial = (outcome == 1 or outcome == -1)
                    else:
                        # Include only hits
                        include_trial = (outcome == 1)

                    if include_trial:
                        # Select valid channels
                        trial_data = trial[valid_idx, :]

                        # Separate by target class
                        if runData.target[i] == 1:
                            trials_class1.append(trial_data)
                        elif runData.target[i] == 2:
                            trials_class2.append(trial_data)

            processed_data[session][task] = {
                'class1': trials_class1,
                'class2': trials_class2,
                'fs': fs
            }

            # Print summary
            total_included = len(trials_class1) + len(trials_class2)
            print(f"{session}-{task}: Included {total_included} trials "
                  f"(Hits: {hit_count}, Misses: {miss_count}, Aborts: {abort_count} [excluded])")

    print("-" * 70)
    return processed_data


def apply_csp_pipeline(processed_data, filter_band=(8, 30), n_components=6):
    """
    Apply CSP to preprocessed data

    Parameters:
    -----------
    processed_data : dict
        Output from preprocess_eeg_data
    filter_band : tuple
        Frequency band for filtering (low, high) Hz
    n_components : int
        Number of CSP components

    Returns:
    --------
    csp_features : dict
        CSP features for each session and task
    csp_models : dict
        Fitted CSP models
    """
    csp_features = {}
    csp_models = {}

    for session, tasks in processed_data.items():
        csp_features[session] = {}
        csp_models[session] = {}

        for task, data in tasks.items():
            print(f"\nProcessing {session} - {task}")
            print(f"Class 1 trials: {len(data['class1'])}")
            print(f"Class 2 trials: {len(data['class2'])}")

            # Initialize CSP
            csp = CSPProcessor(n_components=n_components, filter_band=filter_band)

            # Apply bandpass filter to all trials
            fs = data['fs']
            trials_c1_filtered = [csp.bandpass_filter(t, fs) for t in data['class1']]
            trials_c2_filtered = [csp.bandpass_filter(t, fs) for t in data['class2']]

            # Fit and transform
            features_c1, features_c2 = csp.fit_transform(trials_c1_filtered,
                                                         trials_c2_filtered)

            # Store results
            csp_features[session][task] = {
                'features': np.vstack([features_c1, features_c2]),
                'labels': np.hstack([np.ones(len(features_c1)),
                                    np.zeros(len(features_c2))])
            }
            csp_models[session][task] = csp

            print(f"Features shape: {csp_features[session][task]['features'].shape}")

    return csp_features, csp_models


def create_train_test_split(processed_data, test_size=0.2, random_state=42):
    """
    Create train/test splits for each session and task

    Parameters:
    -----------
    processed_data : dict
        Output from preprocess_eeg_data
    test_size : float
        Proportion of data for testing
    random_state : int
        Random seed for reproducibility

    Returns:
    --------
    train_data, test_data : dicts
        Separate dictionaries for train and test data
    """
    np.random.seed(random_state)
    train_data = {}
    test_data = {}

    for session, tasks in processed_data.items():
        train_data[session] = {}
        test_data[session] = {}

        for task, data in tasks.items():
            n_class1 = len(data['class1'])
            n_class2 = len(data['class2'])

            # Create indices for splitting
            idx_c1 = np.random.permutation(n_class1)
            idx_c2 = np.random.permutation(n_class2)

            split_c1 = int(n_class1 * (1 - test_size))
            split_c2 = int(n_class2 * (1 - test_size))

            # Split class 1
            train_c1 = [data['class1'][i] for i in idx_c1[:split_c1]]
            test_c1 = [data['class1'][i] for i in idx_c1[split_c1:]]

            # Split class 2
            train_c2 = [data['class2'][i] for i in idx_c2[:split_c2]]
            test_c2 = [data['class2'][i] for i in idx_c2[split_c2:]]

            train_data[session][task] = {
                'class1': train_c1,
                'class2': train_c2,
                'fs': data['fs']
            }

            test_data[session][task] = {
                'class1': test_c1,
                'class2': test_c2,
                'fs': data['fs']
            }

            print(f"{session}-{task}: Train C1={len(train_c1)}, C2={len(train_c2)} | "
                  f"Test C1={len(test_c1)}, C2={len(test_c2)}")

    return train_data, test_data


def apply_csp_with_split(train_data, test_data, filter_band=(8, 30), n_components=6):
    """
    Apply CSP to train data and transform test data

    Returns:
    --------
    train_features, test_features : dicts
        CSP features for train and test sets
    csp_models : dict
        Fitted CSP models
    """
    train_features = {}
    test_features = {}
    csp_models = {}

    for session in train_data.keys():
        train_features[session] = {}
        test_features[session] = {}
        csp_models[session] = {}

        for task in train_data[session].keys():
            print(f"\nProcessing {session} - {task}")

            # Initialize CSP
            csp = CSPProcessor(n_components=n_components, filter_band=filter_band)

            # Get training data
            train_d = train_data[session][task]
            test_d = test_data[session][task]
            fs = train_d['fs']

            # Apply bandpass filter to training trials
            train_c1_filt = [csp.bandpass_filter(t, fs) for t in train_d['class1']]
            train_c2_filt = [csp.bandpass_filter(t, fs) for t in train_d['class2']]

            # Fit CSP on training data
            feat_c1_train, feat_c2_train = csp.fit_transform(train_c1_filt, train_c2_filt)

            # Apply bandpass filter to test trials
            test_c1_filt = [csp.bandpass_filter(t, fs) for t in test_d['class1']]
            test_c2_filt = [csp.bandpass_filter(t, fs) for t in test_d['class2']]

            # Transform test data using fitted CSP
            feat_c1_test = np.array([csp.transform(t) for t in test_c1_filt])
            feat_c2_test = np.array([csp.transform(t) for t in test_c2_filt])

            # Store results
            train_features[session][task] = {
                'features': np.vstack([feat_c1_train, feat_c2_train]),
                'labels': np.hstack([np.ones(len(feat_c1_train)),
                                    np.zeros(len(feat_c2_train))])
            }

            test_features[session][task] = {
                'features': np.vstack([feat_c1_test, feat_c2_test]),
                'labels': np.hstack([np.ones(len(feat_c1_test)),
                                    np.zeros(len(feat_c2_test))])
            }

            csp_models[session][task] = csp

            print(f"Train features: {train_features[session][task]['features'].shape}")
            print(f"Test features: {test_features[session][task]['features'].shape}")

    return train_features, test_features, csp_models


# ===== CNN Model for CSP Features =====
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class CSPDataset(Dataset):
    """PyTorch Dataset for CSP features"""
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


class CNNModel_CSP(nn.Module):
    """
    CNN model adapted for CSP features
    Input: (batch_size, n_components, sequence_length)
    Output: (batch_size, 2) for binary classification
    """
    def __init__(self, n_components=6, sequence_length=128, n_classes=2):
        super().__init__()

        self.network = nn.Sequential(
            # ----- CNN Layer 1 -----
            nn.Conv1d(in_channels=n_components, out_channels=64,
                     kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.MaxPool1d(kernel_size=2),   # L: 128 -> 64

            # ----- CNN Layer 2 -----
            nn.Conv1d(in_channels=64, out_channels=128,
                     kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.MaxPool1d(kernel_size=2),   # L: 64 -> 32

            # ----- CNN Layer 3 -----
            nn.Conv1d(in_channels=128, out_channels=256,
                     kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.MaxPool1d(kernel_size=2),   # L: 32 -> 16

            nn.Flatten(),
            # Calculate flattened size: 256 channels * 16 time points
            nn.Linear(256 * 16, 120),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(120, n_classes)
        )

    def forward(self, xb):
        return self.network(xb)


def expand_csp_features(features, target_length=128):
    """
    Expand CSP features to sequence for CNN

    Parameters:
    -----------
    features : array (n_samples, n_components)
    target_length : int
        Desired sequence length

    Returns:
    --------
    expanded : array (n_samples, n_components, target_length)
    """
    n_samples, n_components = features.shape
    expanded = np.zeros((n_samples, n_components, target_length))

    # Replicate features across time dimension
    for i in range(n_samples):
        for j in range(n_components):
            expanded[i, j, :] = features[i, j]

    return expanded


def train_cnn_model(train_features, test_features, session, task,
                   n_epochs=50, batch_size=32, lr=0.001, device='cuda'):
    """
    Train CNN model on CSP features

    Returns:
    --------
    model : trained model
    history : training history
    """
    # Prepare data
    X_train = expand_csp_features(train_features[session][task]['features'])
    y_train = train_features[session][task]['labels'].astype(int)

    X_test = expand_csp_features(test_features[session][task]['features'])
    y_test = test_features[session][task]['labels'].astype(int)

    # Create datasets
    train_dataset = CSPDataset(X_train, y_train)
    test_dataset = CSPDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Initialize model
    n_components = X_train.shape[1]
    model = CNNModel_CSP(n_components=n_components, n_classes=2)
    model = model.to(device)

    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Training loop
    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

    for epoch in range(n_epochs):
        # Training
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        # Testing
        model.eval()
        test_loss = 0
        test_correct = 0
        test_total = 0

        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                test_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                test_total += labels.size(0)
                test_correct += (predicted == labels).sum().item()

        # Record history
        history['train_loss'].append(train_loss / len(train_loader))
        history['train_acc'].append(100 * train_correct / train_total)
        history['test_loss'].append(test_loss / len(test_loader))
        history['test_acc'].append(100 * test_correct / test_total)

        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{n_epochs}] "
                  f"Train Loss: {history['train_loss'][-1]:.4f}, "
                  f"Train Acc: {history['train_acc'][-1]:.2f}%, "
                  f"Test Loss: {history['test_loss'][-1]:.4f}, "
                  f"Test Acc: {history['test_acc'][-1]:.2f}%")

    return model, history


# ==================== VISUALIZATION FUNCTIONS ====================

import seaborn as sns
from mpl_toolkits.axes_grid1 import make_axes_locatable
from sklearn.metrics import confusion_matrix

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10


def plot_raw_eeg_sample(runData, trial_idx=0, duration=2.0):
    """Visualize raw EEG from a single trial"""
    trial = runData.trials[trial_idx]
    fs = runData.fs

    # Use actual trial length or requested duration, whichever is smaller
    actual_duration = trial.shape[1] / fs
    duration = min(duration, actual_duration)
    n_samples = int(duration * fs)
    time = np.arange(n_samples) / fs

    # Select subset of channels for clarity
    channels_to_plot = [0, 5, 10, 15, 20, 25]
    channel_labels = [runData.label[i] for i in channels_to_plot]

    fig, axes = plt.subplots(len(channels_to_plot), 1, figsize=(14, 10), sharex=True)
    fig.suptitle(f'Raw EEG - Trial {trial_idx} (Target: {runData.target[trial_idx]}, Duration: {duration:.2f}s)',
                 fontsize=16, fontweight='bold')

    for i, (ch_idx, ax) in enumerate(zip(channels_to_plot, axes)):
        ax.plot(time, trial[ch_idx, :n_samples], linewidth=0.8, color='steelblue')
        ax.set_ylabel(f'{channel_labels[i]}\n(µV)', fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.axvspan(0, min(0.5, duration), alpha=0.1, color='green', label='Baseline' if i == 0 else '')
        if duration > 0.5:
            ax.axvspan(0.5, duration, alpha=0.1, color='orange', label='Motor Imagery' if i == 0 else '')

    axes[-1].set_xlabel('Time (seconds)', fontsize=12)
    axes[0].legend(loc='upper right')
    plt.tight_layout()
    plt.show()
    return fig


def plot_frequency_spectrum(runData, trial_indices=[0, 1], show_bands=True):
    """Show frequency content of EEG signals"""
    from scipy import signal as sig
    fs = runData.fs

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle('Frequency Spectrum Analysis', fontsize=16, fontweight='bold')

    for idx, trial_idx in enumerate(trial_indices):
        trial = runData.trials[trial_idx]
        target = runData.target[trial_idx]

        freqs, psd = sig.welch(trial[13, :], fs, nperseg=512)

        axes[idx].semilogy(freqs, psd, linewidth=2, color='navy', alpha=0.7)
        axes[idx].set_xlabel('Frequency (Hz)', fontsize=11)
        axes[idx].set_ylabel('Power Spectral Density (µV²/Hz)', fontsize=11)
        axes[idx].set_title(f'Trial {trial_idx} - Target: {"Right/Up" if target == 1 else "Left/Down"}',
                           fontsize=12)
        axes[idx].set_xlim([0, 50])
        axes[idx].grid(True, alpha=0.3)

        if show_bands:
            axes[idx].axvspan(8, 13, alpha=0.2, color='green', label='Mu (8-13 Hz)')
            axes[idx].axvspan(13, 30, alpha=0.2, color='orange', label='Beta (13-30 Hz)')
            axes[idx].legend(loc='upper right')

    plt.tight_layout()
    plt.show()
    return fig


def plot_filtering_effect(trial, fs, filter_band=(8, 30), channel_idx=None):
    """Show effect of bandpass filtering"""
    # If no channel specified, use the middle channel
    if channel_idx is None:
        channel_idx = trial.shape[0] // 2

    # Make sure channel_idx is valid
    if channel_idx >= trial.shape[0]:
        channel_idx = trial.shape[0] // 2
        print(f"Warning: Channel index adjusted to {channel_idx} (middle channel)")

    nyq = 0.5 * fs
    low = filter_band[0] / nyq
    high = filter_band[1] / nyq
    sos = butter(5, [low, high], btype='band', output='sos')
    filtered = sosfiltfilt(sos, trial[channel_idx, :])

    time = np.arange(len(trial[channel_idx, :])) / fs

    # Use actual trial length or 2 seconds, whichever is smaller
    actual_duration = len(trial[channel_idx, :]) / fs
    duration = min(2.0, actual_duration)
    n_samples = int(duration * fs)

    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    fig.suptitle(f'Bandpass Filtering Effect ({filter_band[0]}-{filter_band[1]} Hz) - Channel {channel_idx}',
                 fontsize=16, fontweight='bold')

    axes[0].plot(time[:n_samples], trial[channel_idx, :n_samples],
                linewidth=0.8, color='steelblue')
    axes[0].set_ylabel('Raw EEG (µV)', fontsize=11)
    axes[0].set_title('Original Signal', fontsize=12)
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(time[:n_samples], filtered[:n_samples],
                linewidth=0.8, color='darkgreen')
    axes[1].set_ylabel('Filtered EEG (µV)', fontsize=11)
    axes[1].set_title('After Bandpass Filter (Mu + Beta bands)', fontsize=12)
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(time[:n_samples], trial[channel_idx, :n_samples],
                linewidth=0.8, color='steelblue', alpha=0.5, label='Original')
    axes[2].plot(time[:n_samples], filtered[:n_samples],
                linewidth=0.8, color='darkgreen', label='Filtered')
    axes[2].set_xlabel('Time (seconds)', fontsize=11)
    axes[2].set_ylabel('Amplitude (µV)', fontsize=11)
    axes[2].set_title('Comparison', fontsize=12)
    axes[2].legend(loc='upper right')
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return fig


def plot_csp_patterns(csp_model, channel_labels=None):
    """Visualize CSP spatial filters"""
    filters = csp_model.filters
    n_channels = filters.shape[1]

    # If no labels provided or mismatch, create generic labels
    if channel_labels is None or len(channel_labels) != n_channels:
        channel_labels = [f'Ch{i+1}' for i in range(n_channels)]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(f'CSP Spatial Patterns (Top 3 for each class) - {n_channels} channels',
                 fontsize=16, fontweight='bold')

    axes = axes.flatten()

    for i in range(6):
        pattern = filters[i, :]
        colors = ['red' if p < 0 else 'blue' for p in pattern]
        axes[i].bar(range(len(pattern)), pattern, color=colors, alpha=0.7)
        axes[i].set_xticks(range(len(pattern)))
        axes[i].set_xticklabels(channel_labels, rotation=90, fontsize=8)
        axes[i].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
        axes[i].set_ylabel('Weight', fontsize=10)
        axes[i].grid(True, alpha=0.3, axis='y')

        if i < 3:
            axes[i].set_title(f'CSP Component {i+1} (Class 1)', fontsize=11, color='darkred')
        else:
            axes[i].set_title(f'CSP Component {i+1} (Class 2)', fontsize=11, color='darkblue')

    plt.tight_layout()
    plt.show()
    return fig


def plot_eigenvalue_spectrum(csp_model):
    """Plot CSP eigenvalue spectrum"""
    eigenvalues = csp_model.eigenvalues

    fig, ax = plt.subplots(1, 1, figsize=(12, 6))

    ax.plot(range(len(eigenvalues)), eigenvalues, 'o-',
           linewidth=2, markersize=8, color='navy')
    ax.axhline(y=0.5, color='red', linestyle='--', linewidth=2,
              label='Equal variance (0.5)')

    n_comp = 3
    ax.scatter(range(n_comp), eigenvalues[:n_comp],
              s=200, c='red', marker='o', edgecolors='black',
              linewidth=2, label='Selected (Class 1)', zorder=5)
    ax.scatter(range(len(eigenvalues)-n_comp, len(eigenvalues)),
              eigenvalues[-n_comp:],
              s=200, c='blue', marker='o', edgecolors='black',
              linewidth=2, label='Selected (Class 2)', zorder=5)

    ax.set_xlabel('Component Index', fontsize=12)
    ax.set_ylabel('Eigenvalue (Variance Ratio)', fontsize=12)
    ax.set_title('CSP Eigenvalue Spectrum\n(Higher = More discriminative for Class 1)',
                fontsize=14, fontweight='bold')
    ax.legend(loc='right', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([-0.1, 1.1])

    plt.tight_layout()
    plt.show()
    return fig


def plot_csp_features_distribution(train_features, session='Session1', task='LR'):
    """Visualize distribution of CSP features for both classes"""
    features = train_features[session][task]['features']
    labels = train_features[session][task]['labels']

    class1_features = features[labels == 1]
    class2_features = features[labels == 0]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(f'CSP Feature Distributions - {session} {task}',
                 fontsize=16, fontweight='bold')

    axes = axes.flatten()

    for i in range(6):
        axes[i].hist(class1_features[:, i], bins=20, alpha=0.6,
                    color='red', label='Class 1 (Right/Up)', density=True)
        axes[i].hist(class2_features[:, i], bins=20, alpha=0.6,
                    color='blue', label='Class 2 (Left/Down)', density=True)
        axes[i].set_xlabel(f'CSP Feature {i+1} (log variance)', fontsize=10)
        axes[i].set_ylabel('Density', fontsize=10)
        axes[i].set_title(f'Component {i+1}', fontsize=11)
        axes[i].legend(loc='upper right', fontsize=9)
        axes[i].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return fig

def plot_pipeline_summary(all_results):
    """Summary of results across all sessions and tasks"""
    sessions = list(all_results.keys())
    tasks = ['LR', 'UD']

    data = []
    for session in sessions:
        for task in tasks:
            if task in all_results[session]:
                data.append({
                    'session': session,
                    'task': task,
                    'accuracy': all_results[session][task]['test_acc'][-1]
                })

    fig, ax = plt.subplots(1, 1, figsize=(10, 6))

    x = np.arange(len(data))
    colors = ['steelblue' if d['task'] == 'LR' else 'coral' for d in data]
    bars = ax.bar(x, [d['accuracy'] for d in data], color=colors, alpha=0.8, edgecolor='black')

    ax.set_xlabel('Session - Task', fontsize=12)
    ax.set_ylabel('Test Accuracy (%)', fontsize=12)
    ax.set_title('BCI Classification Performance Summary', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([f"{d['session']}\n{d['task']}" for d in data])
    ax.set_ylim([0, 105])
    ax.axhline(y=50, color='red', linestyle='--', linewidth=2, label='Chance Level (50%)')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3, axis='y')

    for bar, d in zip(bars, data):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 1,
               f'{d["accuracy"]:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.show()
    return fig

def plot_csp_features_2d(train_features, session='Session1', task='LR'):
    """2D scatter plot of most discriminative CSP features"""
    features = train_features[session][task]['features']
    labels = train_features[session][task]['labels']

    class1_features = features[labels == 1]
    class2_features = features[labels == 0]

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle(f'CSP Feature Space - {session} {task}',
                 fontsize=16, fontweight='bold')

    axes[0].scatter(class1_features[:, 0], class1_features[:, 1],
                   c='red', alpha=0.6, s=50, label='Class 1 (Right/Up)', edgecolors='black')
    axes[0].scatter(class2_features[:, 0], class2_features[:, 1],
                   c='blue', alpha=0.6, s=50, label='Class 2 (Left/Down)', edgecolors='black')
    axes[0].set_xlabel('CSP Component 1', fontsize=12)
    axes[0].set_ylabel('CSP Component 2', fontsize=12)
    axes[0].set_title('Components 1 vs 2', fontsize=12)
    axes[0].legend(loc='best')
    axes[0].grid(True, alpha=0.3)

    axes[1].scatter(class1_features[:, -2], class1_features[:, -1],
                   c='red', alpha=0.6, s=50, label='Class 1 (Right/Up)', edgecolors='black')
    axes[1].scatter(class2_features[:, -2], class2_features[:, -1],
                   c='blue', alpha=0.6, s=50, label='Class 2 (Left/Down)', edgecolors='black')
    axes[1].set_xlabel('CSP Component 5', fontsize=12)
    axes[1].set_ylabel('CSP Component 6', fontsize=12)
    axes[1].set_title('Components 5 vs 6', fontsize=12)
    axes[1].legend(loc='best')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return fig


def plot_feature_expansion():
    """Illustrate feature expansion process"""
    csp_features = np.array([2.5, 1.8, 2.1, -0.3, -0.5, -0.2])
    expanded = np.tile(csp_features.reshape(-1, 1), (1, 128))

    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    fig.suptitle('Feature Expansion for CNN Input', fontsize=16, fontweight='bold')

    axes[0].bar(range(6), csp_features, color=['red', 'red', 'red', 'blue', 'blue', 'blue'],
               alpha=0.7, edgecolor='black')
    axes[0].set_xlabel('CSP Component', fontsize=12)
    axes[0].set_ylabel('Log Variance', fontsize=12)
    axes[0].set_title('Original CSP Features (6 values)', fontsize=12)
    axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
    axes[0].grid(True, alpha=0.3, axis='y')
    axes[0].set_xticks(range(6))
    axes[0].set_xticklabels([f'CSP-{i+1}' for i in range(6)])

    im = axes[1].imshow(expanded, aspect='auto', cmap='RdBu_r',
                       interpolation='nearest', vmin=-3, vmax=3)
    axes[1].set_xlabel('Time Steps (Replicated)', fontsize=12)
    axes[1].set_ylabel('CSP Component', fontsize=12)
    axes[1].set_title('Expanded Features (6 × 128) - Ready for CNN', fontsize=12)
    axes[1].set_yticks(range(6))
    axes[1].set_yticklabels([f'CSP-{i+1}' for i in range(6)])

    divider = make_axes_locatable(axes[1])
    cax = divider.append_axes("right", size="2%", pad=0.1)
    plt.colorbar(im, cax=cax, label='Log Variance')

    plt.tight_layout()
    plt.show()
    return fig


def plot_training_history(history, session='Session1', task='LR'):
    """Plot training and validation metrics over epochs"""
    epochs = range(1, len(history['train_loss']) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle(f'Training History - {session} {task}', fontsize=16, fontweight='bold')

    axes[0].plot(epochs, history['train_loss'], 'o-',
                linewidth=2, markersize=4, color='blue', label='Train Loss')
    axes[0].plot(epochs, history['test_loss'], 's-',
                linewidth=2, markersize=4, color='red', label='Test Loss')
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title('Loss vs Epoch', fontsize=13)
    axes[0].legend(loc='upper right', fontsize=11)
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history['train_acc'], 'o-',
                linewidth=2, markersize=4, color='blue', label='Train Accuracy')
    axes[1].plot(epochs, history['test_acc'], 's-',
                linewidth=2, markersize=4, color='red', label='Test Accuracy')
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Accuracy (%)', fontsize=12)
    axes[1].set_title('Accuracy vs Epoch', fontsize=13)
    axes[1].legend(loc='lower right', fontsize=11)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim([0, 105])

    final_train_acc = history['train_acc'][-1]
    final_test_acc = history['test_acc'][-1]
    axes[1].text(0.02, 0.98,
                f'Final Train: {final_train_acc:.2f}%\nFinal Test: {final_test_acc:.2f}%',
                transform=axes[1].transAxes, fontsize=11, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.tight_layout()
    plt.show()
    return fig


def plot_confusion_matrix_bci(model, test_features, session='Session1', task='LR', device='cuda'):
    """Plot confusion matrix for test set predictions"""
    X_test = expand_csp_features(test_features[session][task]['features'])
    y_test = test_features[session][task]['labels'].astype(int)

    model.eval()
    X_test_tensor = torch.FloatTensor(X_test).to(device)

    with torch.no_grad():
        outputs = model(X_test_tensor)
        _, predictions = torch.max(outputs, 1)

    predictions = predictions.cpu().numpy()
    cm = confusion_matrix(y_test, predictions)

    fig, ax = plt.subplots(1, 1, figsize=(8, 6))

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Left/Down', 'Right/Up'],
                yticklabels=['Left/Down', 'Right/Up'],
                cbar_kws={'label': 'Count'}, ax=ax)

    ax.set_xlabel('Predicted Label', fontsize=12)
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_title(f'Confusion Matrix - {session} {task}\nTest Accuracy: {100 * cm.diagonal().sum() / cm.sum():.2f}%',
                fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.show()
    return fig, cm


def calculate_pvc(all_data_dict, session, task):
    """
    Calculate Peak Validation Criterion (PVC) from online classifier performance

    PVC = (# hit trials / # non-aborted trials) * 100

    Parameters:
    -----------
    all_data_dict : dict
        Dictionary containing session and run data
    session : str
        Session name (e.g., 'Session1')
    task : str
        Task name (e.g., 'LR' or 'UD')

    Returns:
    --------
    pvc_score : float
        PVC percentage
    hit_count : int
        Number of hit trials
    miss_count : int
        Number of miss trials
    abort_count : int
        Number of aborted trials
    """
    hit_count = 0
    miss_count = 0
    abort_count = 0

    for runData in all_data_dict[session][task]:
        for outcome in runData.outcome:
            if outcome == 1:  # Hit
                hit_count += 1
            elif outcome == -1:  # Miss
                miss_count += 1
            elif outcome == 0:  # Abort
                abort_count += 1

    non_aborted = hit_count + miss_count
    pvc_score = (hit_count / non_aborted * 100) if non_aborted > 0 else 0

    return pvc_score, hit_count, miss_count, abort_count


def plot_pvc_comparison(all_data_dict, all_results):
    """
    Compare PVC (online classifier) vs offline CNN accuracy

    Parameters:
    -----------
    all_data_dict : dict
        Original data with online classifier outcomes
    all_results : dict
        CNN training results with test accuracies
    """
    data = []

    for session in all_results.keys():
        for task in all_results[session].keys():
            # Calculate PVC from online data
            pvc_score, hits, misses, aborts = calculate_pvc(all_data_dict, session, task)

            # Get offline CNN accuracy
            cnn_accuracy = all_results[session][task]['test_acc'][-1]

            data.append({
                'session': session,
                'task': task,
                'pvc': pvc_score,
                'cnn_acc': cnn_accuracy,
                'hits': hits,
                'misses': misses,
                'aborts': aborts
            })

    # Create comparison plot
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Online Linear Classifier (PVC) vs Offline CNN Performance',
                 fontsize=16, fontweight='bold')

    # Plot 1: Bar chart comparison
    x = np.arange(len(data))
    width = 0.35

    pvc_scores = [d['pvc'] for d in data]
    cnn_scores = [d['cnn_acc'] for d in data]
    labels = [f"{d['session']}\n{d['task']}" for d in data]

    bars1 = axes[0].bar(x - width/2, pvc_scores, width,
                       label='Online PVC', color='coral', alpha=0.8, edgecolor='black')
    bars2 = axes[0].bar(x + width/2, cnn_scores, width,
                       label='Offline CNN', color='steelblue', alpha=0.8, edgecolor='black')

    axes[0].set_xlabel('Session - Task', fontsize=12)
    axes[0].set_ylabel('Accuracy (%)', fontsize=12)
    axes[0].set_title('PVC vs CNN Accuracy Comparison', fontsize=13)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels)
    axes[0].set_ylim([0, 105])
    axes[0].axhline(y=50, color='red', linestyle='--', linewidth=2, label='Chance Level (50%)')
    axes[0].legend(loc='lower right', fontsize=11)
    axes[0].grid(True, alpha=0.3, axis='y')

    # Add value labels on bars
    for bar in bars1:
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height + 1,
                    f'{height:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
    for bar in bars2:
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height + 1,
                    f'{height:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

    # Plot 2: Scatter plot (PVC vs CNN)
    colors_map = {'LR': 'steelblue', 'UD': 'coral'}
    for d in data:
        color = colors_map[d['task']]
        axes[1].scatter(d['pvc'], d['cnn_acc'], s=200, c=color, alpha=0.7,
                       edgecolors='black', linewidth=2, label=d['task'] if d['session'] == 'Session1' else '')
        axes[1].text(d['pvc'] + 1, d['cnn_acc'] + 1,
                    f"{d['session']}\n{d['task']}", fontsize=9, ha='left')

    # Add diagonal line (perfect correlation)
    min_val = min(min(pvc_scores), min(cnn_scores)) - 5
    max_val = max(max(pvc_scores), max(cnn_scores)) + 5
    axes[1].plot([min_val, max_val], [min_val, max_val],
                'k--', linewidth=2, alpha=0.5, label='Perfect Correlation')

    axes[1].set_xlabel('Online PVC (%)', fontsize=12)
    axes[1].set_ylabel('Offline CNN Accuracy (%)', fontsize=12)
    axes[1].set_title('Correlation: Online vs Offline Performance', fontsize=13)
    axes[1].legend(loc='lower right', fontsize=10)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim([min_val, max_val])
    axes[1].set_ylim([min_val, max_val])

    plt.tight_layout()
    plt.show()

    return fig, data


def plot_trial_outcomes_breakdown(all_data_dict):
    """
    Visualize the breakdown of trial outcomes (hit/miss/abort) for each session-task
    """
    data = []

    for session in all_data_dict.keys():
        for task in all_data_dict[session].keys():
            pvc_score, hits, misses, aborts = calculate_pvc(all_data_dict, session, task)
            total = hits + misses + aborts

            data.append({
                'session': session,
                'task': task,
                'hits': hits,
                'misses': misses,
                'aborts': aborts,
                'total': total,
                'pvc': pvc_score
            })

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Online Classifier Trial Outcomes Breakdown', fontsize=16, fontweight='bold')

    # Plot 1: Stacked bar chart
    x = np.arange(len(data))
    labels = [f"{d['session']}\n{d['task']}" for d in data]

    hits = [d['hits'] for d in data]
    misses = [d['misses'] for d in data]
    aborts = [d['aborts'] for d in data]

    axes[0].bar(x, hits, label='Hits', color='green', alpha=0.7, edgecolor='black')
    axes[0].bar(x, misses, bottom=hits, label='Misses', color='red', alpha=0.7, edgecolor='black')
    axes[0].bar(x, aborts, bottom=[h+m for h, m in zip(hits, misses)],
               label='Aborts (ignored in PVC)', color='gray', alpha=0.7, edgecolor='black')

    axes[0].set_xlabel('Session - Task', fontsize=12)
    axes[0].set_ylabel('Number of Trials', fontsize=12)
    axes[0].set_title('Trial Outcomes Distribution', fontsize=13)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels)
    axes[0].legend(loc='upper right', fontsize=11)
    axes[0].grid(True, alpha=0.3, axis='y')

    # Add total count on top
    for i, d in enumerate(data):
        axes[0].text(i, d['total'] + 2, f"n={d['total']}",
                    ha='center', va='bottom', fontsize=10, fontweight='bold')

    # Plot 2: PVC with trial breakdown
    x = np.arange(len(data))
    colors = ['steelblue' if d['task'] == 'LR' else 'coral' for d in data]
    pvcs = [d['pvc'] for d in data]

    bars = axes[1].bar(x, pvcs, color=colors, alpha=0.8, edgecolor='black')

    axes[1].set_xlabel('Session - Task', fontsize=12)
    axes[1].set_ylabel('PVC (%)', fontsize=12)
    axes[1].set_title(' Percent Valid Correct (PVC)\nHits / (Hits + Misses)', fontsize=13)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(labels)
    axes[1].set_ylim([0, 105])
    axes[1].axhline(y=50, color='red', linestyle='--', linewidth=2, label='Chance Level (50%)')
    axes[1].legend(loc='lower right')
    axes[1].grid(True, alpha=0.3, axis='y')

    # Add detailed labels
    for i, (bar, d) in enumerate(zip(bars, data)):
        height = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2., height + 1,
                    f'{height:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
        axes[1].text(bar.get_x() + bar.get_width()/2., height/2,
                    f'{d["hits"]}/{d["hits"]+d["misses"]}',
                    ha='center', va='center', fontsize=9, color='white', fontweight='bold')

    plt.tight_layout()
    plt.show()

    return fig, data


def print_pvc_summary(all_data_dict, all_results):
    """Print detailed PVC and CNN accuracy comparison"""
    print("\n" + "="*80)
    print("PVC (ONLINE LINEAR CLASSIFIER) vs CNN (OFFLINE) PERFORMANCE COMPARISON")
    print("="*80)
    print("\nNote: PVC only considers non-aborted trials (hits + misses)")
    print("      This makes online and offline performance directly comparable\n")

    print(f"{'Session':<12} {'Task':<6} {'Hits':<6} {'Miss':<6} {'Abort':<7} {'PVC':<8} {'CNN Acc':<10} {'Improvement':<12}")
    print("-" * 80)

    for session in all_results.keys():
        for task in all_results[session].keys():
            pvc_score, hits, misses, aborts = calculate_pvc(all_data_dict, session, task)
            cnn_acc = all_results[session][task]['test_acc'][-1]
            improvement = cnn_acc - pvc_score

            print(f"{session:<12} {task:<6} {hits:<6} {misses:<6} {aborts:<7} "
                  f"{pvc_score:>6.2f}%  {cnn_acc:>8.2f}%  {improvement:>+10.2f}%")

    print("-" * 80)

    # Calculate averages
    all_pvc = []
    all_cnn = []
    for session in all_results.keys():
        for task in all_results[session].keys():
            pvc_score, _, _, _ = calculate_pvc(all_data_dict, session, task)
            cnn_acc = all_results[session][task]['test_acc'][-1]
            all_pvc.append(pvc_score)
            all_cnn.append(cnn_acc)

    avg_pvc = np.mean(all_pvc)
    avg_cnn = np.mean(all_cnn)
    avg_improvement = avg_cnn - avg_pvc

    print(f"\n{'AVERAGE':<12} {'ALL':<6} {'':<6} {'':<6} {'':<7} "
          f"{avg_pvc:>6.2f}%  {avg_cnn:>8.2f}%  {avg_improvement:>+10.2f}%")
    print("="*80 + "\n")
    """Summary of results across all sessions and tasks"""
    sessions = list(all_results.keys())
    tasks = ['LR', 'UD']

    data = []
    for session in sessions:
        for task in tasks:
            if task in all_results[session]:
                data.append({
                    'session': session,
                    'task': task,
                    'accuracy': all_results[session][task]['test_acc'][-1]
                })

    fig, ax = plt.subplots(1, 1, figsize=(10, 6))

    x = np.arange(len(data))
    colors = ['steelblue' if d['task'] == 'LR' else 'coral' for d in data]
    bars = ax.bar(x, [d['accuracy'] for d in data], color=colors, alpha=0.8, edgecolor='black')

    ax.set_xlabel('Session - Task', fontsize=12)
    ax.set_ylabel('Test Accuracy (%)', fontsize=12)
    ax.set_title('BCI Classification Performance Summary', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([f"{d['session']}\n{d['task']}" for d in data])
    ax.set_ylim([0, 105])
    ax.axhline(y=50, color='red', linestyle='--', linewidth=2, label='Chance Level (50%)')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3, axis='y')

    for bar, d in zip(bars, data):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 1,
               f'{d["accuracy"]:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.show()
    return fig


# Example usage:
if __name__ == "__main__":
    # ===== LOAD DATA FIRST =====
    print("\n" + "="*70)
    print("LOADING BCI DATA")
    print("="*70)

    # IMPORTANT: Set your data folder path here
    DATA_FOLDER = "/content/drive/MyDrive/Group1 BCI SMR Data"
    # Or use the variable if already defined
    try:
        # Check if all_data_dict already exists
        if 'all_data_dict' not in locals() and 'all_data_dict' not in globals():
            # If not, try to load it
            print("all_data_dict not found. Please provide the data folder path.")
            print("Example: DATA_FOLDER = '/content/drive/MyDrive/Group1 BCI SMR Data'")
            print("\nYou can either:")
            print("1. Run your data loading code first, OR")
            print("2. Uncomment and set DATA_FOLDER below:\n")
            print("# DATA_FOLDER = Path(r'/your/path/here')")
            print("# all_data_dict = load_bci_data(DATA_FOLDER)")
            raise NameError("all_data_dict not defined. Please load data first.")
    except NameError:
        # Try loading if DATA_FOLDER is defined
        if 'DATA_FOLDER' in locals() or 'DATA_FOLDER' in globals():
            all_data_dict = load_bci_data(DATA_FOLDER)
        else:
            raise NameError("Please define DATA_FOLDER or load all_data_dict before running")

    # Step 1: Preprocess and split data
    print("\n" + "="*70)
    print("STEP 1: PREPROCESSING AND SPLITTING DATA")
    print("="*70)

    # IMPORTANT: Set include_misses=True to use same trials as PVC metric
    # This makes CNN performance directly comparable to online classifier PVC
    processed_data = preprocess_eeg_data(all_data_dict, include_misses=True)
    train_data, test_data = create_train_test_split(processed_data, test_size=0.2)

    # Step 2: Apply CSP with train/test split
    print("\n" + "="*70)
    print("STEP 2: APPLYING CSP TO TRAIN AND TEST DATA")
    print("="*70)
    train_features, test_features, csp_models = apply_csp_with_split(
        train_data, test_data,
        filter_band=(8, 30),
        n_components=6
    )

    # ===== VISUALIZATION SECTION =====
    print("\n" + "="*70)
    print("VISUALIZATION SECTION: GENERATING ALL PLOTS")
    print("="*70)

    # Define valid channels
    valid_channels = ['F3','F4','Fc5','Fc3','Fc1','Fcz','Fc2','Fc4','Fc6',
                      'T7','C5','C3','C1','Cz','C2','C4','C6','T8','Cp5',
                      'Cp3','Cp1','Cpz','Cp2','Cp4','Cp6','P3','P4']

    # Generate visualizations for both sessions and both tasks
    for session in ['Session1', 'Session2']:
        for task in ['LR', 'UD']:
            print(f"\n{'='*70}")
            print(f"VISUALIZATIONS FOR {session} - {task}")
            print(f"{'='*70}")

            # Check if this session-task combination exists
            if session not in all_data_dict or task not in all_data_dict[session]:
                print(f"Warning: {session}-{task} not found in data. Skipping...")
                continue

            if len(all_data_dict[session][task]) == 0:
                print(f"Warning: {session}-{task} has no data. Skipping...")
                continue

            # Visualization 1: Raw EEG sample
            print(f"\n=== 1. Raw EEG Sample - {session} {task} ===")
            runData = all_data_dict[session][task][0]
            plot_raw_eeg_sample(runData, trial_idx=5, duration=2.0)

            # Visualization 2: Frequency spectrum
            print(f"\n=== 2. Frequency Spectrum - {session} {task} ===")
            plot_frequency_spectrum(runData, trial_indices=[0, 10])

            # Visualization 3: Filtering effect
            print(f"\n=== 3. Bandpass Filtering Effect - {session} {task} ===")
            if session in processed_data and task in processed_data[session]:
                trial = processed_data[session][task]['class1'][0]
                plot_filtering_effect(trial, fs=300, filter_band=(8, 30))

            # Visualization 4: CSP patterns
            print(f"\n=== 4. CSP Spatial Patterns - {session} {task} ===")
            if session in csp_models and task in csp_models[session]:
                first_run = all_data_dict[session][task][0]
                actual_labels = [label for label in first_run.label if label in valid_channels]
                plot_csp_patterns(csp_models[session][task], actual_labels)

            # Visualization 5: Eigenvalue spectrum
            print(f"\n=== 5. CSP Eigenvalue Spectrum - {session} {task} ===")
            if session in csp_models and task in csp_models[session]:
                plot_eigenvalue_spectrum(csp_models[session][task])

            # Visualization 6: Feature distributions
            print(f"\n=== 6. CSP Feature Distributions - {session} {task} ===")
            if session in train_features and task in train_features[session]:
                plot_csp_features_distribution(train_features, session, task)

            # Visualization 7: 2D feature space
            print(f"\n=== 7. 2D Feature Space - {session} {task} ===")
            if session in train_features and task in train_features[session]:
                plot_csp_features_2d(train_features, session, task)

    # Visualization 8: Feature expansion (only once, it's generic)
    print(f"\n{'='*70}")
    print("=== 8. Feature Expansion Illustration (Generic) ===")
    print(f"{'='*70}")
    plot_feature_expansion()

    # ===== TRAINING SECTION =====
    print("\n" + "="*70)
    print("STEP 3: TRAINING CNN MODELS FOR ALL SESSION-TASK COMBINATIONS")
    print("="*70)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    all_results = {}
    all_models = {}

    for session in train_features.keys():
        all_results[session] = {}
        all_models[session] = {}

        for task in train_features[session].keys():
            print(f"\n{'='*70}")
            print(f"TRAINING: {session} - {task}")
            print(f"{'='*70}")

            # Train the model
            model, history = train_cnn_model(
                train_features, test_features,
                session, task,
                n_epochs=50,
                batch_size=32,
                lr=0.001,
                device=device
            )

            all_results[session][task] = history
            all_models[session][task] = model

            print(f"\n✓ Final Test Accuracy: {history['test_acc'][-1]:.2f}%")

            # Visualization 9: Training history
            print(f"\n=== 9. Training History - {session} {task} ===")
            plot_training_history(history, session, task)

            # Visualization 10: Confusion matrix
            print(f"\n=== 10. Confusion Matrix - {session} {task} ===")
            plot_confusion_matrix_bci(model, test_features, session, task, device)

    # Visualization 11: Overall summary
    print(f"\n{'='*70}")
    print("=== 11. Overall Performance Summary - All Sessions & Tasks ===")
    print(f"{'='*70}")
    plot_pipeline_summary(all_results)

    # Visualization 12: PVC comparison
    print(f"\n{'='*70}")
    print("=== 12. PVC vs CNN Performance Comparison ===")
    print(f"{'='*70}")
    plot_pvc_comparison(all_data_dict, all_results)

    # Visualization 13: Trial outcomes breakdown
    print(f"\n{'='*70}")
    print("=== 13. Trial Outcomes Breakdown (Hit/Miss/Abort) ===")
    print(f"{'='*70}")
    plot_trial_outcomes_breakdown(all_data_dict)

    # Print detailed PVC summary
    print_pvc_summary(all_data_dict, all_results)

    # ===== FINAL SUMMARY =====
    print("\n" + "="*70)
    print("FINAL RESULTS SUMMARY")
    print("="*70)
    for session in all_results.keys():
        print(f"\n{session}:")
        for task in all_results[session].keys():
            final_acc = all_results[session][task]['test_acc'][-1]
            print(f"  {task}: {final_acc:.2f}% test accuracy")

    print("\n" + "="*70)
    print("PIPELINE COMPLETED SUCCESSFULLY!")
    print("="*70)

Output hidden; open in https://colab.research.google.com to view.